<a href="https://colab.research.google.com/github/idoschw3/Corpus2GeoRAG/blob/main/ApplyNER/NER_fine_tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
from collections import Counter, defaultdict
from flair.data import Sentence, Corpus
from flair.models import SequenceTagger
from flair.trainers import ModelTrainer
from IPython.display import display
from tqdm.auto import tqdm
from pathlib import Path
import pandas as pd
import numpy as np
import subprocess
import tempfile
import random
import shutil
import flair
import torch
import time
import json
import sys
import os
import re
import gc

IS_COLAB_VM = (
    "google.colab" in sys.modules
    and os.path.exists("/content")
    and os.environ.get("COLAB_RELEASE_TAG") is not None
)

if IS_COLAB_VM:
    from google.colab import userdata
else:
    userdata = None

### Clone Corpus2GeoRAG Repository

In [19]:
IS_LOCAL = os.name == "nt"

LOCAL_REPO_PATH = Path.home() / "Corpus2GeoRAG"
repo_dir = (
    LOCAL_REPO_PATH
    if IS_LOCAL
    else Path("/content/Corpus2GeoRAG")
)

REPO_PATH = str(repo_dir)

if not repo_dir.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/EtzionR/Corpus2GeoRAG.git",
            REPO_PATH
        ],
        check=True
    )

elif not (repo_dir / ".git").exists():
    raise RuntimeError(
        f"{REPO_PATH} exists but is not a Git checkout."
    )

else:
    # Download remote commit information.
    subprocess.run(
        ["git", "fetch", "origin"],
        cwd=repo_dir,
        check=True
    )

    # Update only if Git can fast-forward safely.
    subprocess.run(
        ["git", "pull", "--ff-only"],
        cwd=repo_dir,
        check=True
    )

os.chdir(REPO_PATH)

print("Runtime:", "Local Windows" if IS_LOCAL else "Colab VM")
print("Working folder:", REPO_PATH)

Runtime: Local Windows
Working folder: C:\Users\idosc\Corpus2GeoRAG


### Cohere cleaning

this step is to clear Cohere extracted entitiies from refrences.

#### Cohere GT upload and cleaning

Uploading and preparing GT

In [20]:
# ============================================================
# 1. FILE PATHS
# ============================================================

repo_dir = Path(REPO_PATH)

evaluation_dir = (
    repo_dir
    / "ApplyNER"
    / "NER_output"
    / "model_evaluation"
)

ground_truth_dir = evaluation_dir / "ground_truth"

cohere_path = ground_truth_dir / "cohere_entities.parquet"
subset_path = evaluation_dir / "evaluation_subset.parquet"
corpus_path = repo_dir / "CollectCorpus" / "DATA.json"

required_files = {
    "Cohere ground truth": cohere_path,
    "Evaluation subset": subset_path,
    "Wikipedia corpus": corpus_path
}

for name, path in required_files.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name} not found: {path}")

# ============================================================
# 2. LOAD ORIGINAL DATA
# ============================================================

cohere_gt_original = pd.read_parquet(cohere_path)
evaluation_df_original = pd.read_parquet(subset_path)

required_entity_columns = {
    "Source", "Index", "Text",
    "word", "entity_group", "start", "end"
}

required_subset_columns = {"Source", "Index", "Text"}

for name, df, required in [
    ("Cohere GT", cohere_gt_original, required_entity_columns),
    ("Evaluation subset", evaluation_df_original, required_subset_columns)
]:
    missing = required - set(df.columns)

    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

    if df[list(required_subset_columns)].isna().any().any():
        raise ValueError(f"{name} contains missing identifiers.")

if evaluation_df_original.empty:
    raise ValueError("Evaluation subset is empty.")

if evaluation_df_original.duplicated(["Source", "Index"]).any():
    raise ValueError("Duplicate Source/Index pairs.")

# ============================================================
# 3. LOAD WIKIPEDIA CORPUS
# ============================================================

with open(corpus_path, "r", encoding="utf-8") as f:
    original_corpus = json.load(f)

if not isinstance(original_corpus, dict):
    raise ValueError("Expected a dictionary of Wikipedia pages.")

required_pages = set(evaluation_df_original["Source"])
missing_pages = required_pages - set(original_corpus)

# ============================================================
# 4. REPRODUCE THE ORIGINAL COHERE PREPROCESSING
# ============================================================

TITLE = "=="
MIN_LENGTH = 5

def reconstruct_paragraphs(page_text):
    """
    EXACT preprocessing used before generating
    the original Cohere ground truth.

    Do not remove References or External links here.
    Their removal belongs to the next Cleaning cell.
    """

    paragraphs = page_text.split("\n")

    return [
        text
        for text in paragraphs
        if not text.startswith(TITLE) and len(text) >= MIN_LENGTH
    ]


reconstructed_paragraphs = {}

for page in required_pages - missing_pages:

    page_text = original_corpus[page]

    if not isinstance(page_text, str):
        raise ValueError(
            f"Unexpected article content type: {page}"
        )

    reconstructed_paragraphs[page] = reconstruct_paragraphs(
        page_text
    )

# ============================================================
# 5. VERIFY ORIGINAL PARAGRAPH INDEX ALIGNMENT
# ============================================================

misaligned_pages = set()
alignment_examples = []

for row in evaluation_df_original[
    ["Source", "Index", "Text"]
].itertuples(index=False):

    page = row.Source

    if page in missing_pages:
        continue

    index = int(row.Index)
    paragraphs = reconstructed_paragraphs[page]

    if (
        index < 0
        or index >= len(paragraphs)
        or paragraphs[index] != row.Text
    ):
        misaligned_pages.add(page)

        if len(alignment_examples) < 10:
            alignment_examples.append({
                "Source": page,
                "Index": index,
                "Expected": row.Text[:150],
                "Reconstructed": (
                    paragraphs[index][:150]
                    if 0 <= index < len(paragraphs)
                    else "Index out of range"
                )
            })

# ============================================================
# 6. VERIFY COHERE AGAINST THE EVALUATION SUBSET
# ============================================================

cohere_paragraphs_original = cohere_gt_original[
    ["Source", "Index", "Text"]
].drop_duplicates()

alignment_check = cohere_paragraphs_original.merge(
    evaluation_df_original[["Source", "Index", "Text"]],
    on=["Source", "Index", "Text"],
    how="left",
    indicator=True
)

cohere_conflict_pages = set(
    alignment_check.loc[
        alignment_check["_merge"] != "both",
        "Source"
    ]
)

# ============================================================
# 7. EXCLUDE ONLY GENUINE CONFLICTS
# ============================================================

excluded_pages = (
    missing_pages
    | misaligned_pages
    | cohere_conflict_pages
)

valid_pages = required_pages - excluded_pages

if not valid_pages:
    raise ValueError("No valid Wikipedia articles remain.")

evaluation_df = (
    evaluation_df_original[
        evaluation_df_original["Source"].isin(valid_pages)
    ]
    .copy()
    .reset_index(drop=True)
)

cohere_gt = (
    cohere_gt_original[
        cohere_gt_original["Source"].isin(valid_pages)
    ]
    .copy()
    .reset_index(drop=True)
)

cohere_paragraphs = cohere_gt[
    ["Source", "Index", "Text"]
].drop_duplicates()

relevant_pages = {
    page: original_corpus[page]
    for page in sorted(valid_pages)
}

original_paragraphs = {
    page: reconstructed_paragraphs[page]
    for page in valid_pages
}

del original_corpus

# ============================================================
# 8. FINAL ALIGNMENT VERIFICATION
# ============================================================

evaluation_keys = set(
    zip(
        evaluation_df["Source"],
        evaluation_df["Index"],
        evaluation_df["Text"]
    )
)

cohere_keys = set(
    zip(
        cohere_paragraphs["Source"],
        cohere_paragraphs["Index"],
        cohere_paragraphs["Text"]
    )
)

if not cohere_keys.issubset(evaluation_keys):
    raise ValueError("Cohere paragraph alignment failed.")

for row in evaluation_df.itertuples(index=False):

    reconstructed = original_paragraphs[row.Source]

    if reconstructed[int(row.Index)] != row.Text:
        raise ValueError(
            f"Final alignment failed: "
            f"{row.Source}, index {row.Index}"
        )

# ============================================================
# 9. SUMMARY
# ============================================================

print("===== COHERE GT PREPARATION =====")

print(f"Original evaluation pages: {len(required_pages):,}")
print(f"Original evaluation paragraphs: {len(evaluation_df_original):,}")
print(f"Original Cohere entities: {len(cohere_gt_original):,}")

print("\n===== CORPUS ALIGNMENT =====")

print(f"Missing pages: {len(missing_pages):,}")
print(f"Modified/misaligned pages: {len(misaligned_pages):,}")
print(f"Cohere/evaluation conflicts: {len(cohere_conflict_pages):,}")
print(f"Total excluded pages: {len(excluded_pages):,}")

print("\n===== PREPARED DATASETS =====")

print(f"Retained Wikipedia pages: {len(valid_pages):,}")
print(f"Retained evaluation paragraphs: {len(evaluation_df):,}")
print(f"Retained Cohere entities: {len(cohere_gt):,}")
print(f"Excluded Cohere entities: {len(cohere_gt_original) - len(cohere_gt):,}")

if excluded_pages:
    print("\n===== EXCLUDED ARTICLES =====")

    for page in sorted(excluded_pages)[:20]:
        reasons = []

        if page in missing_pages:
            reasons.append("Missing from corpus")

        if page in misaligned_pages:
            reasons.append("Paragraph content/index mismatch")

        if page in cohere_conflict_pages:
            reasons.append("Cohere/evaluation mismatch")

        print(f"{page}: {', '.join(reasons)}")

if alignment_examples:
    print("\n===== ALIGNMENT MISMATCH EXAMPLES =====")
    display(pd.DataFrame(alignment_examples))

print("\nPreparation completed.")
print("Original datasets remain unchanged.")

display(cohere_gt.head(10))

===== COHERE GT PREPARATION =====
Original evaluation pages: 191
Original evaluation paragraphs: 6,565
Original Cohere entities: 7,326

===== CORPUS ALIGNMENT =====
Missing pages: 30
Modified/misaligned pages: 0
Cohere/evaluation conflicts: 0
Total excluded pages: 30

===== PREPARED DATASETS =====
Retained Wikipedia pages: 161
Retained evaluation paragraphs: 5,416
Retained Cohere entities: 6,799
Excluded Cohere entities: 527

===== EXCLUDED ARTICLES =====
Anti-Russian sentiment: Missing from corpus
Bridge over Saint Volodymyr Descent: Missing from corpus
Chernobyl exclusion zone: Missing from corpus
Cyber Partisans: Missing from corpus
Davyd Zhvania: Missing from corpus
Freedom on Fire: Ukraine's Fight for Freedom: Missing from corpus
Fyodor Berezin: Missing from corpus
Great Construction: Missing from corpus
Intermarium: Missing from corpus
Ivano-Frankivsk Oblast: Missing from corpus
Kamianka Sich: Missing from corpus
Kharkiv State Academy of Culture: Missing from corpus
Mejlis of the

,entity_group,score,word,start,end,Source,Index,Text
0,ORG,NaN,Ternopil Battalion,4,22,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
1,ORG,NaN,Special Tasks Patrol Police,51,78,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
2,ORG,NaN,Ministry of Internal Affairs of Ukraine,95,134,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
3,LOC,NaN,Ternopil,162,170,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
4,MISC,NaN,War in Donbass,218,232,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
5,LOC,NaN,Ternopil Oblast,100,115,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."
6,MISC,NaN,Euromaidan,216,226,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."
7,PER,NaN,Vyacheslav Kizilov,296,314,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."
8,MISC,NaN,ATO,420,423,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."
9,MISC,NaN,ATO,491,494,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."


Cleaning

In [21]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

TITLE = "=="
MIN_LENGTH = 5

STOP_HEADINGS = {"references", "external links"}

heading_pattern = re.compile(
    r"(?P<marks>={2,})\s*(?P<heading>.*?)\s*(?P=marks)"
)

output_path = (
    Path(REPO_PATH)
    / "ApplyNER"
    / "NER_output"
    / "model_evaluation"
    / "ground_truth"
    / "cohere_entities_cleaned.parquet"
)

# ============================================================
# 2. FIND CUTOFFS USING ORIGINAL PARAGRAPH INDICES
# ============================================================

cutoff_positions = {}
cutoff_headings = {}
cutoff_counts = Counter()

for page, page_text in relevant_pages.items():

    original_index = 0
    cutoff = len(original_paragraphs[page])
    cutoff_heading = None

    for text in page_text.split("\n"):

        heading_match = heading_pattern.fullmatch(text.strip())

        if heading_match:
            heading_text = heading_match.group("heading").strip()

            normalized_heading = " ".join(
                heading_text.casefold().split()
            )

            if normalized_heading in STOP_HEADINGS:
                cutoff = original_index
                cutoff_heading = normalized_heading
                cutoff_counts[normalized_heading] += 1
                break

        # EXACT original Cohere preprocessing.
        # Count only lines that originally received an Index.
        if not text.startswith(TITLE) and len(text) >= MIN_LENGTH:
            original_index += 1

    cutoff_positions[page] = cutoff
    cutoff_headings[page] = cutoff_heading

# ============================================================
# 3. IDENTIFY PARAGRAPHS AFTER THE CUTOFF
# ============================================================

excluded_paragraphs = set()

for row in evaluation_df[
    ["Source", "Index", "Text"]
].itertuples(index=False):

    page = row.Source
    index = int(row.Index)

    # Alignment must still match the original Cohere data.
    if original_paragraphs[page][index] != row.Text:
        raise ValueError(
            f"Alignment failed: {page}, index {index}"
        )

    if index >= cutoff_positions[page]:
        excluded_paragraphs.add((page, index))

# ============================================================
# 4. REMOVE ONLY ENTITIES FROM EXCLUDED PARAGRAPHS
# ============================================================

remove_mask = pd.Series(
    [
        (source, int(index)) in excluded_paragraphs
        for source, index in zip(
            cohere_gt["Source"],
            cohere_gt["Index"]
        )
    ],
    index=cohere_gt.index,
    dtype=bool
)

removed_entities = cohere_gt.loc[remove_mask].copy()

cohere_gt_cleaned = (
    cohere_gt.loc[~remove_mask]
    .copy()
    .reset_index(drop=True)
)

# ============================================================
# 5. VERIFY THE RESULTS
# ============================================================

original_count = len(cohere_gt)
removed_count = len(removed_entities)
cleaned_count = len(cohere_gt_cleaned)

if original_count != removed_count + cleaned_count:
    raise ValueError("Entity counts do not match.")

for row in cohere_gt_cleaned[
    ["Source", "Index", "Text", "word", "start", "end"]
].itertuples(index=False):

    if (row.Source, int(row.Index)) in excluded_paragraphs:
        raise ValueError("An excluded paragraph remains.")

    start = int(row.start)
    end = int(row.end)

    if row.Text[start:end] != row.word:
        raise ValueError(
            f"Entity offset mismatch: "
            f"{row.Source}, index {row.Index}"
        )

# ============================================================
# 6. SAVE SEPARATE CLEANED GT
# ============================================================

output_path.parent.mkdir(parents=True, exist_ok=True)

save_required = True

if output_path.exists():
    existing = pd.read_parquet(output_path)

    if existing.equals(cohere_gt_cleaned):
        save_required = False
        print("Existing cleaned GT already matches.")

if save_required:
    temporary_path = output_path.with_suffix(".tmp")

    cohere_gt_cleaned.to_parquet(
        temporary_path,
        engine="pyarrow",
        compression="zstd",
        index=False
    )

    os.replace(temporary_path, output_path)

    print("Cleaned Cohere GT saved successfully.")

# ============================================================
# 7. CLEANING SUMMARY
# ============================================================

removed_percentage = (
    removed_count / original_count * 100
    if original_count else 0
)

print("\n===== COHERE CLEANING SUMMARY =====")

print(f"Original entities in valid articles: {original_count:,}")
print(f"Removed reference entities: {removed_count:,}")
print(f"Cleaned entities: {cleaned_count:,}")
print(f"Entities removed: {removed_percentage:.2f}%")

print(f"Evaluation paragraphs excluded: {len(excluded_paragraphs):,}")
print(f"Pages with References cutoff: {cutoff_counts['references']:,}")
print(f"Pages with External links cutoff: {cutoff_counts['external links']:,}")

print("\n===== REMOVED ENTITIES BY CUTOFF =====")

if not removed_entities.empty:

    removed_by_heading = (
        removed_entities["Source"]
        .map(cutoff_headings)
        .value_counts()
    )

    for heading, count in removed_by_heading.items():
        print(f"{heading}: {count:,} entities")

    print("\n===== REMOVED ENTITY PREVIEW =====")

    display(
        removed_entities[
            ["Source", "Index", "word", "entity_group"]
        ].head(20)
    )

print("\n===== CLEANED GT PREVIEW =====")
display(cohere_gt_cleaned.head(10))

print(f"\nOutput: {output_path}")
print("Original Cohere GT was not modified.")

Existing cleaned GT already matches.

===== COHERE CLEANING SUMMARY =====
Original entities in valid articles: 6,799
Removed reference entities: 72
Cleaned entities: 6,727
Entities removed: 1.06%
Evaluation paragraphs excluded: 138
Pages with References cutoff: 158
Pages with External links cutoff: 1

===== REMOVED ENTITIES BY CUTOFF =====
references: 72 entities

===== REMOVED ENTITY PREVIEW =====


,Source,Index,word,entity_group
470,Murder of Pentecostals in Sloviansk,28,"Karpitsky, Nikolai Nikolayevich",PER
471,Murder of Pentecostals in Sloviansk,28,Donbas,LOC
472,Murder of Pentecostals in Sloviansk,28,Eastern Ukraine,LOC
473,Murder of Pentecostals in Sloviansk,28,Mission Eurasia,ORG
474,Murder of Pentecostals in Sloviansk,29,Богдан Красовский,PER
475,Murder of Pentecostals in Sloviansk,29,Славянске,LOC
476,Murder of Pentecostals in Sloviansk,29,Славянск,LOC
477,Murder of Pentecostals in Sloviansk,30,Дмитрий Фионик,PER
478,Murder of Pentecostals in Sloviansk,30,Славянске,LOC
479,Murder of Pentecostals in Sloviansk,30,ООО «Фокус Медиа»,ORG



===== CLEANED GT PREVIEW =====


,entity_group,score,word,start,end,Source,Index,Text
0,ORG,NaN,Ternopil Battalion,4,22,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
1,ORG,NaN,Special Tasks Patrol Police,51,78,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
2,ORG,NaN,Ministry of Internal Affairs of Ukraine,95,134,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
3,LOC,NaN,Ternopil,162,170,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
4,MISC,NaN,War in Donbass,218,232,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
5,LOC,NaN,Ternopil Oblast,100,115,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."
6,MISC,NaN,Euromaidan,216,226,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."
7,PER,NaN,Vyacheslav Kizilov,296,314,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."
8,MISC,NaN,ATO,420,423,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."
9,MISC,NaN,ATO,491,494,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."



Output: C:\Users\idosc\Corpus2GeoRAG\ApplyNER\NER_output\model_evaluation\ground_truth\cohere_entities_cleaned.parquet
Original Cohere GT was not modified.


Evaluation

In [24]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_FLERT = "flair/ner-english-large"
LABELS = ("PER", "ORG", "LOC", "MISC")

GPU_BATCH_SIZE = 2
CPU_BATCH_SIZE = 1

FLERT_SETTINGS = {
    "model": MODEL_FLERT,
    "tokenizer": "flair_default",
    "input_unit": "whole_paragraph",
    "label_type": "ner",
    "context_mode": "independent_paragraphs"
}

evaluation_dir = (
    Path(REPO_PATH)
    / "ApplyNER"
    / "NER_output"
    / "model_evaluation"
)

gt_dir = evaluation_dir / "ground_truth"
model_dir = evaluation_dir / "NER_models_evaluation"

original_gt_path = gt_dir / "cohere_entities.parquet"
cleaned_gt_path = gt_dir / "cohere_entities_cleaned.parquet"
subset_path = evaluation_dir / "evaluation_subset.parquet"

cohere_raw_path = gt_dir / "cohere_command_a_03_2025_raw.jsonl"

# Separate outputs for the large FLERT model.
original_flert_path = model_dir / "flert_original_entities.parquet"
cleaned_flert_path = model_dir / "flert_cleaned_entities.parquet"

original_flert_raw = model_dir / "flert_original_raw.jsonl"
cleaned_flert_raw = model_dir / "flert_cleaned_raw.jsonl"

ENTITY_COLUMNS = [
    "entity_group", "score", "word",
    "start", "end", "Source", "Index", "Text"
]

required_files = {
    "Original Cohere GT": original_gt_path,
    "Cleaned Cohere GT": cleaned_gt_path,
    "Evaluation subset": subset_path,
    "Cohere checkpoint": cohere_raw_path
}

for name, path in required_files.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name} not found: {path}")

for name in (
    "valid_pages",
    "excluded_paragraphs",
    "evaluation_df",
    "original_paragraphs"
):
    if name not in globals():
        raise RuntimeError(
            f"Missing {name}. Run preparation and cleaning first."
        )

valid_page_set = set(valid_pages)

excluded_keys = {
    (str(source), int(index))
    for source, index in excluded_paragraphs
}

model_dir.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. LOAD COHERE DATA
# ============================================================

gt_original = pd.read_parquet(original_gt_path)
gt_cleaned = pd.read_parquet(cleaned_gt_path)
subset = pd.read_parquet(subset_path)

required_entity_columns = set(ENTITY_COLUMNS) - {"score"}

for name, df in [
    ("Original Cohere", gt_original),
    ("Cleaned Cohere", gt_cleaned)
]:
    missing = required_entity_columns - set(df.columns)

    if missing:
        raise ValueError(
            f"{name} missing columns: {missing}"
        )

if not {"Source", "Index", "Text"}.issubset(subset.columns):
    raise ValueError("Evaluation subset missing columns.")

if subset.duplicated(["Source", "Index"]).any():
    raise ValueError("Duplicate evaluation paragraph indices.")

# ============================================================
# 3. LOAD COHERE COMPLETION RECORDS
# ============================================================

def paragraph_key(record):
    return (
        str(record["Source"]),
        int(record["Index"]),
        str(record["Text"])
    )


def load_checkpoints(path):
    records = {}

    if not path.exists():
        return records

    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, 1):

            if not line.strip():
                continue

            try:
                record = json.loads(line)
                key = paragraph_key(record)

            except (ValueError, KeyError, TypeError) as exc:
                raise ValueError(
                    f"Invalid checkpoint: {path.name}, "
                    f"line {line_number}"
                ) from exc

            # Latest saved response is authoritative.
            records[key] = record

    return records


def cohere_is_complete(record):

    if record.get("model") != "command-a-03-2025":
        return False

    if "COMPLETE" not in str(
        record.get("finish_reason", "")
    ).upper():
        return False

    raw_text = record.get("raw_response")

    if not isinstance(raw_text, str) or not raw_text:
        raw_text = "".join(
            block.get("text", "")
            for block in (record.get("response_content") or [])
            if (
                isinstance(block, dict)
                and block.get("type") == "text"
                and isinstance(block.get("text"), str)
            )
        ).strip()

    try:
        parsed = json.loads(raw_text)
    except (ValueError, TypeError):
        return False

    return (
        isinstance(parsed, dict)
        and isinstance(parsed.get("entities"), list)
    )


cohere_records = load_checkpoints(cohere_raw_path)

cohere_completed = {
    key
    for key, record in cohere_records.items()
    if cohere_is_complete(record)
}

# ============================================================
# 4. DEFINE ORIGINAL AND CLEANED PARAGRAPHS
# ============================================================

evaluation_keys = {
    paragraph_key(row._asdict())
    for row in subset[
        ["Source", "Index", "Text"]
    ].itertuples(index=False)
}

prepared_keys = {
    paragraph_key(row._asdict())
    for row in evaluation_df[
        ["Source", "Index", "Text"]
    ].itertuples(index=False)
}

original_scope = (
    evaluation_keys
    & prepared_keys
    & cohere_completed
)

original_scope = {
    key for key in original_scope
    if key[0] in valid_page_set
}

cleaned_scope = {
    key for key in original_scope
    if (key[0], key[1]) not in excluded_keys
}

if not original_scope:
    raise ValueError("No original paragraphs available.")

if not cleaned_scope:
    raise ValueError("No cleaned paragraphs available.")

# Confirm both runs receive correctly aligned text.
for source, index, text in original_scope:

    if original_paragraphs[source][index] != text:
        raise ValueError(
            f"Original paragraph alignment failed: "
            f"{source}, index {index}"
        )

print("===== EVALUATION POPULATION =====")
print(f"Evaluation subset: {len(evaluation_keys):,}")
print(f"Cohere completed: {len(cohere_completed):,}")
print(f"Valid original paragraphs: {len(original_scope):,}")
print(f"Cleaned paragraphs: {len(cleaned_scope):,}")
print(
    "Reference paragraphs removed: "
    f"{len(original_scope - cleaned_scope):,}"
)

# ============================================================
# 5. VERIFY COHERE CLEANING
# ============================================================

def annotation_counts(df):

    return Counter(
        (
            str(r.Source),
            int(r.Index),
            str(r.Text),
            int(r.start),
            int(r.end),
            str(r.entity_group),
            str(r.word)
        )
        for r in df.itertuples(index=False)
    )


expected_cleaned = gt_original.loc[
    [
        (
            str(source) in valid_page_set
            and (str(source), int(index)) not in excluded_keys
        )
        for source, index in zip(
            gt_original["Source"],
            gt_original["Index"]
        )
    ]
]

if annotation_counts(expected_cleaned) != annotation_counts(gt_cleaned):
    raise ValueError(
        "Cleaned Cohere GT does not match expected filtering."
    )

print("Cohere cleaning validation passed.")

# ============================================================
# 6. INITIALIZE FLERT ONLY WHEN INFERENCE IS NEEDED
# ============================================================

flert_model = None

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

batch_size = (
    GPU_BATCH_SIZE
    if device.type == "cuda"
    else CPU_BATCH_SIZE
)


def get_flert_model():
    global flert_model

    if flert_model is None:

        import flair
        from flair.models import SequenceTagger

        flair.device = device

        flert_model = SequenceTagger.load(MODEL_FLERT)
        flert_model.to(device)
        flert_model.eval()

        print("\nFLERT loaded.")
        print("Model:", MODEL_FLERT)
        print("Device:", device)
        print("Batch size:", batch_size)

    return flert_model

# ============================================================
# 7. RUN FLERT WITH RESUMABLE CHECKPOINTS
# ============================================================

def run_flert(scope, checkpoint_path, output_path, run_name):

    from flair.data import Sentence

    paragraphs = sorted(scope)
    saved = load_checkpoints(checkpoint_path)

    # Validate previous checkpoints before resuming.
    for key, record in saved.items():

        if key not in scope:
            raise ValueError(
                f"{run_name}: checkpoint contains "
                "unexpected paragraphs."
            )

        if record.get("settings") != FLERT_SETTINGS:
            raise ValueError(
                f"{run_name}: checkpoint model/settings mismatch."
            )

        if not isinstance(record.get("entities"), list):
            raise ValueError(
                f"{run_name}: invalid checkpoint entities."
            )

    remaining = [
        key for key in paragraphs
        if key not in saved
    ]

    print(f"\n===== {run_name.upper()} =====")
    print(f"Paragraphs: {len(paragraphs):,}")
    print(f"Already checkpointed: {len(saved):,}")
    print(f"New inference required: {len(remaining):,}")

    # Do not silently replace existing results.
    if output_path.exists() and remaining:
        raise RuntimeError(
            f"{run_name}: an existing Parquet is present, "
            "but some checkpoints are missing. "
            "Inspect the files before resuming."
        )

    if remaining:

        model = get_flert_model()
        checkpoint_path.parent.mkdir(
            parents=True, exist_ok=True
        )

        started = time.perf_counter()

        with tqdm(
            total=len(remaining),
            desc=run_name,
            unit="paragraph"
        ) as progress:

            for offset in range(
                0, len(remaining), batch_size
            ):

                batch = remaining[
                    offset:offset + batch_size
                ]

                sentences = [
                    Sentence(key[2], use_tokenizer=True)
                    for key in batch
                ]

                nonempty = [
                    sentence
                    for sentence in sentences
                    if len(sentence) > 0
                ]

                if nonempty:

                    with torch.no_grad():
                        model.predict(
                            nonempty,
                            mini_batch_size=batch_size,
                            verbose=False,
                            embedding_storage_mode="none"
                        )

                for key, sentence in zip(
                    batch, sentences, strict=True
                ):

                    entities = []

                    for span in sentence.get_spans("ner"):

                        label = span.get_label("ner")

                        start = int(span.start_position)
                        end = int(span.end_position)

                        if not (
                            0 <= start < end <= len(key[2])
                        ):
                            raise ValueError(
                                f"Invalid FLERT span: "
                                f"{key[0]}, {key[1]}"
                            )

                        entities.append({
                            "entity_group": str(label.value),
                            "score": float(label.score),
                            "word": key[2][start:end],
                            "start": start,
                            "end": end
                        })

                    entities.sort(
                        key=lambda item: (
                            item["start"],
                            item["end"]
                        )
                    )

                    record = {
                        "Source": key[0],
                        "Index": key[1],
                        "Text": key[2],
                        "settings": FLERT_SETTINGS,
                        "entities": entities
                    }

                    # Immediate checkpoint, including zero
                    # entity paragraphs.
                    with checkpoint_path.open(
                        "a", encoding="utf-8"
                    ) as f:

                        f.write(
                            json.dumps(
                                record,
                                ensure_ascii=False,
                                allow_nan=False
                            ) + "\n"
                        )

                        f.flush()
                        os.fsync(f.fileno())

                    saved[key] = record
                    progress.update(1)

                del sentences, nonempty

        print(
            f"Inference time: "
            f"{time.perf_counter() - started:.1f} seconds"
        )

    else:
        print("All predictions already available.")

    if set(saved) != scope:
        raise ValueError(
            f"{run_name}: incomplete inference. "
            "Rerun the cell to resume."
        )

    # ========================================================
    # EXPORT PARQUET
    # ========================================================

    rows = []

    for key in paragraphs:

        for entity in saved[key]["entities"]:

            rows.append({
                **entity,
                "Source": key[0],
                "Index": key[1],
                "Text": key[2]
            })

    result = pd.DataFrame(
        rows,
        columns=ENTITY_COLUMNS
    ).astype({
        "entity_group": "string",
        "score": "float64",
        "word": "string",
        "start": "int64",
        "end": "int64",
        "Source": "string",
        "Index": "int64",
        "Text": "string"
    })

    if output_path.exists():

        existing = pd.read_parquet(output_path)

        try:
            pd.testing.assert_frame_equal(
                existing.reset_index(drop=True),
                result.reset_index(drop=True),
                check_dtype=False
            )
        except AssertionError as exc:
            raise ValueError(
                f"{run_name}: existing Parquet differs "
                "from checkpoint predictions."
            ) from exc

        print("Existing Parquet verified.")

    else:

        temporary = output_path.with_suffix(".tmp")

        result.to_parquet(
            temporary,
            engine="pyarrow",
            compression="zstd",
            index=False
        )

        os.replace(temporary, output_path)

        print("Predictions saved:", output_path.name)

    return result

# ============================================================
# 8. FIRST RUN: ORIGINAL DATA
# ============================================================

flert_original = run_flert(
    original_scope,
    original_flert_raw,
    original_flert_path,
    "FLERT Original"
)

# ============================================================
# 9. SECOND RUN: CLEANED DATA
# ============================================================

flert_cleaned = run_flert(
    cleaned_scope,
    cleaned_flert_raw,
    cleaned_flert_path,
    "FLERT Cleaned"
)

# ============================================================
# 10. BUILD EXACT-MATCH ENTITY SETS
# ============================================================

def entity_set(df, scope, name):

    entities = set()

    for row in df.itertuples(index=False):

        key = (
            str(row.Source),
            int(row.Index),
            str(row.Text)
        )

        if key not in scope:
            continue

        start = int(row.start)
        end = int(row.end)
        label = str(row.entity_group)
        word = str(row.word)

        if label not in LABELS:
            raise ValueError(
                f"{name}: unexpected label {label}"
            )

        if not (0 <= start < end <= len(key[2])):
            raise ValueError(
                f"{name}: invalid character offsets"
            )

        if key[2][start:end] != word:
            raise ValueError(
                f"{name}: entity text mismatch"
            )

        entity = (*key, start, end, label)

        if entity in entities:
            raise ValueError(
                f"{name}: duplicate entity annotation"
            )

        entities.add(entity)

    return entities


original_truth = entity_set(
    gt_original, original_scope, "Original Cohere"
)

cleaned_truth = entity_set(
    gt_cleaned, cleaned_scope, "Cleaned Cohere"
)

original_predictions = entity_set(
    flert_original, original_scope, "Original FLERT"
)

cleaned_predictions = entity_set(
    flert_cleaned, cleaned_scope, "Cleaned FLERT"
)

# Control: original FLERT predictions restricted to
# paragraphs retained by the cleaning process.
original_on_cleaned = entity_set(
    flert_original, cleaned_scope, "Original FLERT control"
)

# ============================================================
# 11. CALCULATE PRECISION, RECALL AND F1
# ============================================================

def calculate_metrics(truth, predictions):

    tp_set = truth & predictions
    fp_set = predictions - truth
    fn_set = truth - predictions

    def scores(tp, fp, fn):

        precision = (
            tp / (tp + fp) if tp + fp else 0.0
        )

        recall = (
            tp / (tp + fn) if tp + fn else 0.0
        )

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        return (
            precision * 100,
            recall * 100,
            f1 * 100
        )

    tp = len(tp_set)
    fp = len(fp_set)
    fn = len(fn_set)

    precision, recall, micro_f1 = scores(tp, fp, fn)

    per_label = []

    for label in LABELS:

        label_tp = sum(
            entity[-1] == label
            for entity in tp_set
        )

        label_fp = sum(
            entity[-1] == label
            for entity in fp_set
        )

        label_fn = sum(
            entity[-1] == label
            for entity in fn_set
        )

        p, r, f1 = scores(
            label_tp, label_fp, label_fn
        )

        per_label.append({
            "Label": label,
            "Support": label_tp + label_fn,
            "Precision (%)": p,
            "Recall (%)": r,
            "F1 (%)": f1
        })

    macro_f1 = sum(
        row["F1 (%)"] for row in per_label
    ) / len(LABELS)

    overall = {
        "GT entities": len(truth),
        "Predicted entities": len(predictions),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision (%)": precision,
        "Recall (%)": recall,
        "Micro F1 (%)": micro_f1,
        "Macro F1 (%)": macro_f1
    }

    return overall, per_label

# ============================================================
# 12. RUN THREE EVALUATIONS
# ============================================================

original_metrics, original_labels = calculate_metrics(
    original_truth,
    original_predictions
)

control_metrics, control_labels = calculate_metrics(
    cleaned_truth,
    original_on_cleaned
)

cleaned_metrics, cleaned_labels = calculate_metrics(
    cleaned_truth,
    cleaned_predictions
)

# ============================================================
# 13. OVERALL RESULTS
# ============================================================

comparison = pd.DataFrame([
    {
        "Evaluation": "FLERT Original / Original GT",
        "Paragraphs": len(original_scope),
        **original_metrics
    },
    {
        "Evaluation": "FLERT Original / Cleaned GT (control)",
        "Paragraphs": len(cleaned_scope),
        **control_metrics
    },
    {
        "Evaluation": "FLERT Cleaned / Cleaned GT",
        "Paragraphs": len(cleaned_scope),
        **cleaned_metrics
    }
]).set_index("Evaluation")

print("\n===== FLERT OVERALL EVALUATION =====")

display(comparison.round(2))

# ============================================================
# 14. PERFORMANCE CHANGE
# ============================================================

metric_columns = [
    "Precision (%)",
    "Recall (%)",
    "Micro F1 (%)",
    "Macro F1 (%)"
]

changes = pd.DataFrame({
    "Metric": metric_columns,
    "Before (%)": [
        original_metrics[m] for m in metric_columns
    ],
    "After (%)": [
        cleaned_metrics[m] for m in metric_columns
    ],
    "Difference (pp)": [
        cleaned_metrics[m] - original_metrics[m]
        for m in metric_columns
    ],
    "Rerun effect (pp)": [
        cleaned_metrics[m] - control_metrics[m]
        for m in metric_columns
    ]
})

print("\n===== FLERT PERFORMANCE CHANGE =====")

display(changes.round(2))

# ============================================================
# 15. PER-ENTITY-TYPE METRICS
# ============================================================

per_label_comparison = pd.concat([
    pd.DataFrame(original_labels).assign(
        Evaluation="Original FLERT / Original GT"
    ),
    pd.DataFrame(control_labels).assign(
        Evaluation="Original FLERT / Cleaned GT (control)"
    ),
    pd.DataFrame(cleaned_labels).assign(
        Evaluation="Cleaned FLERT / Cleaned GT"
    )
], ignore_index=True)

print("\n===== FLERT PER-ENTITY-TYPE METRICS =====")

display(
    per_label_comparison[
        [
            "Evaluation",
            "Label",
            "Support",
            "Precision (%)",
            "Recall (%)",
            "F1 (%)"
        ]
    ].round(2)
)

# ============================================================
# 16. COMPARE PREDICTIONS ON SHARED PARAGRAPHS
# ============================================================

matching = original_on_cleaned & cleaned_predictions
original_only = original_on_cleaned - cleaned_predictions
cleaned_only = cleaned_predictions - original_on_cleaned

print("\n===== ORIGINAL VS CLEANED FLERT RUN =====")

print(f"Matching predictions: {len(matching):,}")
print(f"Original-only predictions: {len(original_only):,}")
print(f"Cleaned-only predictions: {len(cleaned_only):,}")

if original_only or cleaned_only:

    difference_rows = []

    for status, entities in [
        ("Original only", original_only),
        ("Cleaned only", cleaned_only)
    ]:
        for source, index, text, start, end, label in sorted(entities):
            difference_rows.append({
                "Status": status,
                "Source": source,
                "Index": index,
                "Entity": text[start:end],
                "Label": label,
                "Start": start,
                "End": end
            })

    display(pd.DataFrame(difference_rows).head(30))

# ============================================================
# 17. FINAL SUMMARY
# ============================================================

print("\n===== SAVED FLERT RESULTS =====")
print("Original predictions:", original_flert_path)
print("Cleaned predictions:", cleaned_flert_path)
print("Original checkpoint:", original_flert_raw)
print("Cleaned checkpoint:", cleaned_flert_raw)

print("\nEvaluation completed.")
print("Both FLERT inference runs are recorded separately.")
print("Original Cohere and Flair files remain unchanged.")
print("No Cohere API calls were made.")


===== EVALUATION POPULATION =====
Evaluation subset: 6,565
Cohere completed: 979
Valid original paragraphs: 904
Cleaned paragraphs: 880
Reference paragraphs removed: 24
Cohere cleaning validation passed.

===== FLERT ORIGINAL =====
Paragraphs: 904
Already checkpointed: 0
New inference required: 904
2026-10-24 09:26:36,130 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, S-ORG, S-MISC, B-PER, E-PER, S-LOC, B-ORG, E-ORG, I-PER, S-PER, B-MISC, I-MISC, E-MISC, I-ORG, B-LOC, E-LOC, I-LOC, <START>, <STOP>

FLERT loaded.
Model: flair/ner-english-large
Device: cuda:0
Batch size: 2


FLERT Original: 100%|██████████| 904/904 [00:21<00:00, 42.78paragraph/s]


Inference time: 21.1 seconds
Predictions saved: flert_original_entities.parquet

===== FLERT CLEANED =====
Paragraphs: 880
Already checkpointed: 0
New inference required: 880


FLERT Cleaned: 100%|██████████| 880/880 [00:20<00:00, 43.15paragraph/s]


Inference time: 20.4 seconds
Predictions saved: flert_cleaned_entities.parquet

===== FLERT OVERALL EVALUATION =====


...


===== FLERT PER-ENTITY-TYPE METRICS =====


,Evaluation,Label,Support,Precision (%),Recall (%),F1 (%)
0,Original FLERT / Original GT,PER,917,88.12,95.42,91.62
1,Original FLERT / Original GT,ORG,1362,76.91,74.60,75.74
2,Original FLERT / Original GT,LOC,3003,91.33,94.37,92.83
3,Original FLERT / Original GT,MISC,1517,57.86,82.53,68.02
4,Original FLERT / Cleaned GT (control),PER,904,88.43,95.58,91.87
5,Original FLERT / Cleaned GT (control),ORG,1335,77.11,74.68,75.88
6,Original FLERT / Cleaned GT (control),LOC,2991,91.36,94.38,92.85
7,Original FLERT / Cleaned GT (control),MISC,1497,58.04,82.70,68.21
8,Cleaned FLERT / Cleaned GT,PER,904,88.71,95.58,92.01
9,Cleaned FLERT / Cleaned GT,ORG,1335,76.34,74.68,75.50



===== ORIGINAL VS CLEANED FLERT RUN =====
Matching predictions: 7,346
Original-only predictions: 147
Cleaned-only predictions: 144


,Status,Source,Index,Entity,Label,Start,End
0,Original only,Belarusian partisan movement (2020–present),7,Beriev A-50U,MISC,40,52
1,Original only,Black Sea Fleet,0,Chernomorskiy flot,ORG,60,78
2,Original only,Black Sea Fleet,12,Black Sea Fleet,ORG,294,309
3,Original only,Black Sea Fleet,28,Soviet,ORG,325,331
4,Original only,Black Sea Fleet,28,Black Sea Fleet,ORG,332,347
5,Original only,Black Sea Fleet,28,Soviet,LOC,382,388
6,Original only,Black Sea Fleet,28,Black Sea,LOC,389,398
7,Original only,Black Sea Fleet,31,Coastal Troops,ORG,8,22
8,Original only,Black Sea Fleet,31,Naval Aviation,ORG,27,41
9,Original only,Black Sea Fleet,41,51st independent Coastal Missile Regiment,ORG,0,41



===== SAVED FLERT RESULTS =====
Original predictions: C:\Users\idosc\Corpus2GeoRAG\ApplyNER\NER_output\model_evaluation\NER_models_evaluation\flert_original_entities.parquet
Cleaned predictions: C:\Users\idosc\Corpus2GeoRAG\ApplyNER\NER_output\model_evaluation\NER_models_evaluation\flert_cleaned_entities.parquet
Original checkpoint: C:\Users\idosc\Corpus2GeoRAG\ApplyNER\NER_output\model_evaluation\NER_models_evaluation\flert_original_raw.jsonl
Cleaned checkpoint: C:\Users\idosc\Corpus2GeoRAG\ApplyNER\NER_output\model_evaluation\NER_models_evaluation\flert_cleaned_raw.jsonl

Evaluation completed.
Both FLERT inference runs are recorded separately.
Original Cohere and Flair files remain unchanged.
No Cohere API calls were made.


### FLERT error identification and fine tune

Error identification

In [27]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
VALIDATION_FRACTION = 0.20

evaluation_dir = (
    Path(REPO_PATH)
    / "ApplyNER"
    / "NER_output"
    / "model_evaluation"
)

gt_path = (
    evaluation_dir
    / "ground_truth"
    / "cohere_entities_cleaned.parquet"
)

flert_dir = evaluation_dir / "NER_models_evaluation"

flert_path = flert_dir / "flert_cleaned_entities.parquet"
checkpoint_path = flert_dir / "flert_cleaned_raw.jsonl"

output_dir = evaluation_dir / "fine_tuning"

split_path = output_dir / "flert_article_split.parquet"
review_path = output_dir / "flert_train_review.csv"
disagreement_path = (
    output_dir / "flert_train_disagreements.csv"
)

for path in [gt_path, flert_path, checkpoint_path]:
    if not path.is_file():
        raise FileNotFoundError(path)

output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. LOAD DATA
# ============================================================

gt = pd.read_parquet(gt_path)
predictions = pd.read_parquet(flert_path)

required_columns = {
    "Source", "Index", "Text",
    "start", "end", "word", "entity_group"
}

for name, df in [
    ("Cohere GT", gt),
    ("FLERT predictions", predictions)
]:
    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"{name}: missing {missing}")

# Read the completed FLERT checkpoint, including paragraphs
# where the model predicted zero entities.
completed = {}

with checkpoint_path.open("r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, 1):
        if not line.strip():
            continue

        record = json.loads(line)

        if record.get("settings", {}).get("model") != (
            "flair/ner-english-large"
        ):
            raise ValueError(
                f"Unexpected model at checkpoint line {line_number}"
            )

        key = (
            str(record["Source"]),
            int(record["Index"]),
            str(record["Text"])
        )

        completed[key] = record

scope = set(completed)

if not scope:
    raise ValueError("No completed FLERT paragraphs found.")

# ============================================================
# 3. CREATE OR RESTORE ARTICLE-LEVEL SPLIT
# ============================================================

if split_path.exists():

    split_df = pd.read_parquet(split_path)

    required_split_columns = {
        "Source", "Index", "Text", "Split"
    }

    if not required_split_columns.issubset(split_df.columns):
        raise ValueError("Existing split file is invalid.")

    saved_scope = {
        (str(r.Source), int(r.Index), str(r.Text))
        for r in split_df.itertuples(index=False)
    }

    if saved_scope != scope:
        raise ValueError(
            "Existing split doesn't match current FLERT data. "
            "The split was not overwritten."
        )

    print("Existing train/validation split restored.")

else:

    page_counts = Counter(
        source for source, index, text in scope
    )

    pages = sorted(page_counts)

    if len(pages) < 2:
        raise ValueError(
            "At least two articles are needed for the split."
        )

    validation_page_count = max(
        1, round(len(pages) * VALIDATION_FRACTION)
    )

    validation_page_count = min(
        validation_page_count,
        len(pages) - 1
    )

    target_count = len(scope) * VALIDATION_FRACTION

    best_pages = None
    best_difference = float("inf")

    # Choose an article-level split whose paragraph count
    # is as close as possible to 20% validation.
    for attempt in range(300):

        shuffled = pages.copy()
        random.Random(SEED + attempt).shuffle(shuffled)

        candidate = set(shuffled[:validation_page_count])

        candidate_count = sum(
            page_counts[page] for page in candidate
        )

        difference = abs(candidate_count - target_count)

        if difference < best_difference:
            best_difference = difference
            best_pages = candidate

    split_df = pd.DataFrame(
        [
            {
                "Source": source,
                "Index": index,
                "Text": text,
                "Split": (
                    "validation"
                    if source in best_pages
                    else "train"
                )
            }
            for source, index, text in sorted(scope)
        ]
    )

    split_df.to_parquet(
        split_path,
        engine="pyarrow",
        compression="zstd",
        index=False
    )

# Verify that an article never belongs to both splits.
split_by_source = split_df.groupby("Source")["Split"].nunique()

if (split_by_source > 1).any():
    raise ValueError(
        "Article leakage detected between train and validation."
    )

split_lookup = {
    (str(r.Source), int(r.Index), str(r.Text)): r.Split
    for r in split_df.itertuples(index=False)
}

train_scope = {
    key for key in scope
    if split_lookup[key] == "train"
}

validation_scope = scope - train_scope

if not train_scope or not validation_scope:
    raise ValueError("Training or validation split is empty.")

# ============================================================
# 4. INDEX ENTITY ANNOTATIONS
# ============================================================

def index_entities(df, name):

    indexed = defaultdict(dict)

    for row in df.itertuples(index=False):

        key = (
            str(row.Source),
            int(row.Index),
            str(row.Text)
        )

        if key not in scope:
            continue

        start = int(row.start)
        end = int(row.end)
        label = str(row.entity_group)
        word = str(row.word)

        if not (0 <= start < end <= len(key[2])):
            raise ValueError(
                f"{name}: invalid offsets for {key[0]}"
            )

        if key[2][start:end] != word:
            raise ValueError(
                f"{name}: text/offset mismatch for {key[0]}"
            )

        entity_key = (start, end, label)

        if entity_key in indexed[key]:
            raise ValueError(
                f"{name}: duplicate annotation for {key[0]}"
            )

        indexed[key][entity_key] = {
            "word": word,
            "score": (
                float(row.score)
                if hasattr(row, "score")
                and pd.notna(row.score)
                else None
            )
        }

    return indexed


gt_index = index_entities(gt, "Cohere")
flert_index = index_entities(predictions, "FLERT")

# ============================================================
# 5. CLASSIFY AGREEMENTS AND DISAGREEMENTS
# ============================================================

review_rows = []

def add_review_row(status, key, gold=None, pred=None):

    source, index, text = key

    pred_data = (
        flert_index[key].get(pred)
        if pred is not None else None
    )

    review_rows.append({
        "Source": source,
        "Index": index,
        "Text": text,
        "Split": split_lookup[key],
        "Status": status,
        "Cohere_word": (
            text[gold[0]:gold[1]]
            if gold is not None else None
        ),
        "Cohere_label": (
            gold[2] if gold is not None else None
        ),
        "Cohere_start": (
            gold[0] if gold is not None else None
        ),
        "Cohere_end": (
            gold[1] if gold is not None else None
        ),
        "FLERT_word": (
            text[pred[0]:pred[1]]
            if pred is not None else None
        ),
        "FLERT_label": (
            pred[2] if pred is not None else None
        ),
        "FLERT_start": (
            pred[0] if pred is not None else None
        ),
        "FLERT_end": (
            pred[1] if pred is not None else None
        ),
        "FLERT_score": (
            pred_data["score"]
            if pred_data is not None else None
        ),
        "check": (
            "agreement_unverified"
            if status == "exact_match"
            else "pending"
        ),
        "notes": ""
    })


for key in sorted(scope):

    gold = set(gt_index[key])
    pred = set(flert_index[key])

    # A. Perfect agreement on span and label.
    exact = gold & pred

    for entity in sorted(exact):
        add_review_row(
            "exact_match", key, entity, entity
        )

    remaining_gold = gold - exact
    remaining_pred = pred - exact

    # B. Same span, different label.
    for g in sorted(list(remaining_gold)):

        matches = [
            p for p in sorted(remaining_pred)
            if p[:2] == g[:2]
        ]

        if matches:
            p = matches[0]

            add_review_row(
                "label_disagreement", key, g, p
            )

            remaining_gold.remove(g)
            remaining_pred.remove(p)

    # C. Overlapping spans with different boundaries.
    overlap_candidates = []

    for g in remaining_gold:
        for p in remaining_pred:

            overlap = max(
                0,
                min(g[1], p[1]) - max(g[0], p[0])
            )

            if overlap > 0:
                overlap_candidates.append(
                    (overlap, g, p)
                )

    overlap_candidates.sort(
        key=lambda item: (
            -item[0],
            item[1],
            item[2]
        )
    )

    for overlap, g, p in overlap_candidates:

        if g not in remaining_gold:
            continue

        if p not in remaining_pred:
            continue

        status = (
            "boundary_disagreement"
            if g[2] == p[2]
            else "boundary_and_label_disagreement"
        )

        add_review_row(status, key, g, p)

        remaining_gold.remove(g)
        remaining_pred.remove(p)

    # D. Entities FLERT extracted but Cohere did not.
    for p in sorted(remaining_pred):
        add_review_row(
            "extra_flert_prediction", key, None, p
        )

    # E. Entities Cohere extracted but FLERT missed.
    for g in sorted(remaining_gold):
        add_review_row(
            "missed_by_flert", key, g, None
        )

review_df = pd.DataFrame(review_rows)

# ============================================================
# 6. PREPARE THE TRAINING REVIEW FILES
# ============================================================

train_review = (
    review_df[
        review_df["Split"] == "train"
    ]
    .copy()
    .reset_index(drop=True)
)

train_disagreements = (
    train_review[
        train_review["Status"] != "exact_match"
    ]
    .copy()
    .reset_index(drop=True)
)

# Never overwrite manually reviewed files.
for dataframe, path in [
    (train_review, review_path),
    (train_disagreements, disagreement_path)
]:
    if path.exists():
        print(f"Existing review file preserved: {path.name}")
    else:
        dataframe.to_csv(
            path,
            index=False,
            encoding="utf-8-sig"
        )
        print(f"Saved: {path.name}")

# ============================================================
# 7. SUMMARY
# ============================================================

print("\n===== ARTICLE-LEVEL SPLIT =====")

print(f"Total paragraphs: {len(scope):,}")
print(f"Training paragraphs: {len(train_scope):,}")
print(f"Validation paragraphs: {len(validation_scope):,}")

print(
    "Training articles: "
    f"{len({key[0] for key in train_scope}):,}"
)
print(
    "Validation articles: "
    f"{len({key[0] for key in validation_scope}):,}"
)

print("\n===== TRAINING ERROR ANALYSIS =====")

print(f"Review rows: {len(train_review):,}")
print(f"Disagreement rows: {len(train_disagreements):,}")

print("\nStatus breakdown:")
print(
    train_review["Status"]
    .value_counts()
    .to_string()
)

print("\n===== DISAGREEMENT PREVIEW =====")

display(
    train_disagreements[
        [
            "Source",
            "Index",
            "Status",
            "Cohere_word",
            "Cohere_label",
            "FLERT_word",
            "FLERT_label",
            "FLERT_score",
            "check"
        ]
    ].head(30)
)

print("\n===== OUTPUT FILES =====")
print("Split:", split_path)
print("Full training review:", review_path)
print("Training disagreements:", disagreement_path)

Saved: flert_train_review.csv
Saved: flert_train_disagreements.csv

===== ARTICLE-LEVEL SPLIT =====
Total paragraphs: 880
Training paragraphs: 707
Validation paragraphs: 173
Training articles: 24
Validation articles: 6

===== TRAINING ERROR ANALYSIS =====
Review rows: 6,548
Disagreement rows: 1,419

Status breakdown:
Status
exact_match                        5129
extra_flert_prediction              765
boundary_disagreement               236
label_disagreement                  214
boundary_and_label_disagreement     116
missed_by_flert                      88

===== DISAGREEMENT PREVIEW =====


,Source,Index,Status,Cohere_word,Cohere_label,FLERT_word,FLERT_label,FLERT_score,check
0,144th Guards Motor Rifle Division,0,boundary_disagreement,144th Guards Yelnya Red Banner Order of Suvoro...,ORG,144th Guards Yelnya Red Banner Order,ORG,0.999990,pending
1,144th Guards Motor Rifle Division,0,extra_flert_prediction,NaN,NaN,Suvorov Motor Rifle Division,ORG,0.999988,pending
2,144th Guards Motor Rifle Division,0,extra_flert_prediction,NaN,NaN,Краснознамённая,ORG,0.758807,pending
3,144th Guards Motor Rifle Division,2,label_disagreement,Soviet,ORG,Soviet,MISC,0.999928,pending
4,144th Guards Motor Rifle Division,2,label_disagreement,Baltic,LOC,Baltic,MISC,0.989480,pending
5,144th Guards Motor Rifle Division,2,label_disagreement,Soviet,ORG,Soviet,MISC,0.999978,pending
6,144th Guards Motor Rifle Division,2,label_disagreement,Baltic,LOC,Baltic,MISC,0.914941,pending
7,144th Guards Motor Rifle Division,2,boundary_and_label_disagreement,Order of the Red Banner,MISC,of the Red Banner,ORG,0.923439,pending
8,144th Guards Motor Rifle Division,2,extra_flert_prediction,NaN,NaN,Order,MISC,0.913410,pending
9,144th Guards Motor Rifle Division,2,extra_flert_prediction,NaN,NaN,87th,ORG,0.741927,pending



===== OUTPUT FILES =====
Split: C:\Users\idosc\Corpus2GeoRAG\ApplyNER\NER_output\model_evaluation\fine_tuning\flert_article_split.parquet
Full training review: C:\Users\idosc\Corpus2GeoRAG\ApplyNER\NER_output\model_evaluation\fine_tuning\flert_train_review.csv
Training disagreements: C:\Users\idosc\Corpus2GeoRAG\ApplyNER\NER_output\model_evaluation\fine_tuning\flert_train_disagreements.csv


In [28]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
BATCH_PARAGRAPHS = 50
MAX_INITIAL_PER_ARTICLE = 2

review_dir = (
    Path(REPO_PATH)
    / "ApplyNER"
    / "NER_output"
    / "model_evaluation"
    / "fine_tuning"
)

input_path = review_dir / "flert_train_review.csv"
output_path = review_dir / "flert_review_batch_01.csv"

if not input_path.is_file():
    raise FileNotFoundError(input_path)

# Never overwrite manual review work.
if output_path.exists():
    raise FileExistsError(
        f"Review batch already exists: {output_path}\n"
        "Existing annotations were preserved."
    )

# ============================================================
# 2. LOAD REVIEW DATA
# ============================================================

review = pd.read_csv(
    input_path,
    keep_default_na=False
)

required = {
    "Source", "Index", "Text", "Status",
    "Cohere_word", "Cohere_label",
    "FLERT_word", "FLERT_label",
    "check", "notes"
}

missing = required - set(review.columns)

if missing:
    raise ValueError(f"Missing review columns: {missing}")

paragraph_cols = ["Source", "Index", "Text"]

disagreements = review.loc[
    review["Status"] != "exact_match"
].copy()

# ============================================================
# 3. PRIORITIZE DISAGREEMENT TYPES
# ============================================================

weights = {
    "boundary_and_label_disagreement": 4,
    "label_disagreement": 3,
    "boundary_disagreement": 2,
    "missed_by_flert": 2,
    "extra_flert_prediction": 1
}

disagreements["priority"] = (
    disagreements["Status"]
    .map(weights)
    .fillna(0)
)

paragraph_priority = (
    disagreements
    .groupby(paragraph_cols, as_index=False)
    .agg(
        disagreement_count=("Status", "size"),
        priority_score=("priority", "sum")
    )
)

paragraph_priority = paragraph_priority.sort_values(
    ["priority_score", "disagreement_count"],
    ascending=False
)

# ============================================================
# 4. SAMPLE ACROSS DIFFERENT ARTICLES
# ============================================================

selected = []
selected_keys = set()

# Start with up to two paragraphs per article.
for source, group in paragraph_priority.groupby(
    "Source", sort=True
):

    sampled = group.head(MAX_INITIAL_PER_ARTICLE)

    for row in sampled.itertuples(index=False):
        key = (row.Source, row.Index, row.Text)

        if key not in selected_keys:
            selected.append(key)
            selected_keys.add(key)

# If needed, add more high-priority paragraphs.
if len(selected) < BATCH_PARAGRAPHS:

    for row in paragraph_priority.itertuples(index=False):

        if len(selected) >= BATCH_PARAGRAPHS:
            break

        key = (row.Source, row.Index, row.Text)

        if key not in selected_keys:
            selected.append(key)
            selected_keys.add(key)

selected = selected[:BATCH_PARAGRAPHS]
selected_keys = set(selected)

# ============================================================
# 5. INCLUDE ALL ANNOTATIONS FOR SELECTED PARAGRAPHS
# ============================================================

batch_mask = [
    (row.Source, row.Index, row.Text) in selected_keys
    for row in review.itertuples(index=False)
]

review_batch = review.loc[batch_mask].copy()

# Disagreements appear first, followed by exact matches.
review_batch["review_order"] = (
    review_batch["Status"] == "exact_match"
).astype(int)

review_batch = (
    review_batch.sort_values(
        ["Source", "Index", "review_order", "Status"]
    )
    .drop(columns="review_order")
    .reset_index(drop=True)
)

# ============================================================
# 6. SAVE REVIEW BATCH
# ============================================================

review_batch.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 7. SUMMARY
# ============================================================

batch_disagreements = review_batch.loc[
    review_batch["Status"] != "exact_match"
]

print("===== FLERT REVIEW BATCH 01 =====")

print(f"Selected paragraphs: {len(selected_keys):,}")
print(f"Articles represented: {review_batch['Source'].nunique():,}")
print(f"Total annotation rows: {len(review_batch):,}")
print(f"Disagreements: {len(batch_disagreements):,}")

print("\nDisagreement categories:")
print(
    batch_disagreements["Status"]
    .value_counts()
    .to_string()
)

print("\n===== REVIEW PREVIEW =====")

display(
    batch_disagreements[
        [
            "Source",
            "Index",
            "Status",
            "Cohere_word",
            "Cohere_label",
            "FLERT_word",
            "FLERT_label",
            "FLERT_score",
            "check"
        ]
    ].head(20)
)

print("\nReview file:", output_path)
print("Original files remain unchanged.")

===== FLERT REVIEW BATCH 01 =====
Selected paragraphs: 50
Articles represented: 23
Total annotation rows: 801
Disagreements: 300

Disagreement categories:
Status
extra_flert_prediction             159
label_disagreement                  56
boundary_disagreement               36
boundary_and_label_disagreement     25
missed_by_flert                     24

===== REVIEW PREVIEW =====


,Source,Index,Status,Cohere_word,Cohere_label,FLERT_word,FLERT_label,FLERT_score,check
0,144th Guards Motor Rifle Division,2,boundary_and_label_disagreement,Order of the Red Banner,MISC,of the Red Banner,ORG,0.9234393388032913,pending
1,144th Guards Motor Rifle Division,2,extra_flert_prediction,,,Order,MISC,0.9134103655815125,pending
2,144th Guards Motor Rifle Division,2,extra_flert_prediction,,,87th,ORG,0.7419265508651733,pending
3,144th Guards Motor Rifle Division,2,label_disagreement,Soviet,ORG,Soviet,MISC,0.9999278783798218,pending
4,144th Guards Motor Rifle Division,2,label_disagreement,Baltic,LOC,Baltic,MISC,0.9894803762435913,pending
5,144th Guards Motor Rifle Division,2,label_disagreement,Soviet,ORG,Soviet,MISC,0.9999779462814331,pending
6,144th Guards Motor Rifle Division,2,label_disagreement,Baltic,LOC,Baltic,MISC,0.9149414896965027,pending
14,144th Guards Motor Rifle Division,38,boundary_and_label_disagreement,Simferopol,LOC,Simferopol Red Banner,MISC,0.9998474717140198,pending
15,144th Guards Motor Rifle Division,38,boundary_and_label_disagreement,Order of Suvorov,MISC,of Suvorov,ORG,0.7534103989601135,pending
16,144th Guards Motor Rifle Division,38,extra_flert_prediction,,,Order,PER,0.8393046259880066,pending



Review file: C:\Users\idosc\Corpus2GeoRAG\ApplyNER\NER_output\model_evaluation\fine_tuning\flert_review_batch_01.csv
Original files remain unchanged.


In [30]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_NAME = "flair/ner-english-large"
BATCH_SIZE = 2

model_dir = (
    Path(REPO_PATH)
    / "ApplyNER"
    / "NER_output"
    / "model_evaluation"
    / "NER_models_evaluation"
)

gt_dir = model_dir.parent / "ground_truth"

runs = {
    "original": {
        "source": model_dir / "flert_original_raw.jsonl",
        "checkpoint": model_dir / "flert_original_article_context_raw.jsonl",
        "output": model_dir / "flert_original_article_context_entities.parquet",
        "gt": gt_dir / "cohere_entities.parquet",
    },
    "cleaned": {
        "source": model_dir / "flert_cleaned_raw.jsonl",
        "checkpoint": model_dir / "flert_cleaned_article_context_raw.jsonl",
        "output": model_dir / "flert_cleaned_article_context_entities.parquet",
        "gt": gt_dir / "cohere_entities_cleaned.parquet",
    },
}

SETTINGS = {
    "model": MODEL_NAME,
    "context": "article",
    "order": "Source_then_Index",
    "version": 1,
}

for config in runs.values():
    for name in ("source", "gt"):
        if not config[name].is_file():
            raise FileNotFoundError(config[name])

# ============================================================
# 2. CHECKPOINT HELPERS
# ============================================================

def record_key(record):
    return (
        str(record["Source"]),
        int(record["Index"]),
        str(record["Text"]),
    )


def load_records(path):
    records = {}

    if not path.exists():
        return records

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, 1):
            if not line.strip():
                continue

            record = json.loads(line)
            key = record_key(record)

            if key in records:
                raise ValueError(
                    f"Duplicate checkpoint key in {path.name}, "
                    f"line {line_number}"
                )

            records[key] = record

    return records

# ============================================================
# 3. LOAD MODEL
# ============================================================

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

flair.device = device

model = SequenceTagger.load(MODEL_NAME)
model.to(device)
model.eval()

print("===== ARTICLE-CONTEXT FLERT =====")
print("Model:", MODEL_NAME)
print("Device:", device)
print("Mini batch size:", BATCH_SIZE)

# ============================================================
# 4. RUN ARTICLE-LEVEL INFERENCE
# ============================================================

def evaluate_run(run_name, config):

    source_records = load_records(config["source"])
    scope = set(source_records)

    saved = load_records(config["checkpoint"])

    if not scope:
        raise ValueError(f"{run_name}: empty scope.")

    for key, record in saved.items():

        if key not in scope:
            raise ValueError(
                f"{run_name}: unexpected checkpoint paragraph."
            )

        if record.get("settings") != SETTINGS:
            raise ValueError(
                f"{run_name}: incompatible checkpoint settings."
            )

    groups = defaultdict(list)

    for key in scope:
        groups[key[0]].append(key)

    for source in groups:
        groups[source].sort(key=lambda key: key[1])

    remaining = scope - set(saved)

    print(f"\n===== {run_name.upper()} =====")
    print(f"Articles: {len(groups):,}")
    print(f"Paragraphs: {len(scope):,}")
    print(f"Already checkpointed: {len(saved):,}")
    print(f"Remaining: {len(remaining):,}")

    if config["output"].exists() and remaining:
        raise RuntimeError(
            f"{run_name}: output exists but checkpoints "
            "are incomplete. Existing output was preserved."
        )

    started = time.perf_counter()

    with tqdm(
        total=len(remaining),
        desc=f"FLERT {run_name}",
        unit="paragraph",
    ) as progress:

        for source in sorted(groups):

            keys = groups[source]

            # Reprocess the complete article whenever any
            # of its paragraphs still need prediction.
            if all(key in saved for key in keys):
                continue

            sentences = [
                Sentence(key[2], use_tokenizer=True)
                for key in keys
            ]

            # Explicit article boundary.
            sentences[0].is_document_boundary = True

            with torch.no_grad():
                model.predict(
                    sentences,
                    mini_batch_size=BATCH_SIZE,
                    verbose=False,
                    embedding_storage_mode="none",
                )

            for key, sentence in zip(keys, sentences):

                if key in saved:
                    continue

                entities = []

                for span in sentence.get_spans("ner"):

                    label = span.get_label("ner")
                    start = int(span.start_position)
                    end = int(span.end_position)

                    if not (0 <= start < end <= len(key[2])):
                        raise ValueError(
                            f"Invalid span: {source}, {key[1]}"
                        )

                    entities.append({
                        "entity_group": str(label.value),
                        "score": float(label.score),
                        "word": key[2][start:end],
                        "start": start,
                        "end": end,
                    })

                record = {
                    "Source": key[0],
                    "Index": key[1],
                    "Text": key[2],
                    "settings": SETTINGS,
                    "entities": entities,
                }

                with config["checkpoint"].open(
                    "a", encoding="utf-8"
                ) as file:
                    file.write(
                        json.dumps(
                            record,
                            ensure_ascii=False,
                            allow_nan=False,
                        ) + "\n"
                    )
                    file.flush()

                saved[key] = record
                progress.update(1)

    if set(saved) != scope:
        raise ValueError(
            f"{run_name}: inference is incomplete."
        )

    print(
        "Inference time:",
        round(time.perf_counter() - started, 1),
        "seconds",
    )

    # ========================================================
    # 5. EXPORT PREDICTIONS
    # ========================================================

    rows = []

    for key in sorted(scope):
        for entity in saved[key]["entities"]:
            rows.append({
                **entity,
                "Source": key[0],
                "Index": key[1],
                "Text": key[2],
            })

    columns = [
        "entity_group", "score", "word", "start", "end",
        "Source", "Index", "Text"
    ]

    predictions = pd.DataFrame(rows, columns=columns)

    if config["output"].exists():

        existing = pd.read_parquet(config["output"])

        if not existing.equals(predictions):
            raise ValueError(
                f"{run_name}: existing Parquet differs "
                "from checkpoint data."
            )

    else:
        predictions.to_parquet(
            config["output"],
            engine="pyarrow",
            compression="zstd",
            index=False,
        )

    print("Saved:", config["output"].name)

    return scope, predictions

# ============================================================
# 6. PERFORM BOTH RUNS
# ============================================================

original_scope, original_predictions_df = evaluate_run(
    "original",
    runs["original"],
)

cleaned_scope, cleaned_predictions_df = evaluate_run(
    "cleaned",
    runs["cleaned"],
)

if not cleaned_scope.issubset(original_scope):
    raise ValueError(
        "Cleaned paragraphs are not a subset of original."
    )

# ============================================================
# 7. EXACT-MATCH EVALUATION
# ============================================================

def entity_set(df, scope, name):

    entities = set()

    for row in df.itertuples(index=False):

        key = (
            str(row.Source),
            int(row.Index),
            str(row.Text),
        )

        if key not in scope:
            continue

        start = int(row.start)
        end = int(row.end)
        label = str(row.entity_group)

        if not (0 <= start < end <= len(key[2])):
            raise ValueError(f"{name}: invalid offsets.")

        if key[2][start:end] != str(row.word):
            raise ValueError(f"{name}: word/offset mismatch.")

        item = (*key, start, end, label)

        if item in entities:
            raise ValueError(f"{name}: duplicate entity.")

        entities.add(item)

    return entities


def calculate_scores(gt, pred):

    tp = len(gt & pred)
    fp = len(pred - gt)
    fn = len(gt - pred)

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    return {
        "GT entities": len(gt),
        "Predicted entities": len(pred),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision (%)": precision * 100,
        "Recall (%)": recall * 100,
        "Micro F1 (%)": f1 * 100,
    }


original_gt = entity_set(
    pd.read_parquet(runs["original"]["gt"]),
    original_scope,
    "Original GT",
)

cleaned_gt = entity_set(
    pd.read_parquet(runs["cleaned"]["gt"]),
    cleaned_scope,
    "Cleaned GT",
)

original_pred = entity_set(
    original_predictions_df,
    original_scope,
    "Original FLERT",
)

cleaned_pred = entity_set(
    cleaned_predictions_df,
    cleaned_scope,
    "Cleaned FLERT",
)

original_control = {
    entity for entity in original_pred
    if entity[:3] in cleaned_scope
}

results = pd.DataFrame([
    {
        "Evaluation": "Original / Original GT",
        "Paragraphs": len(original_scope),
        **calculate_scores(original_gt, original_pred),
    },
    {
        "Evaluation": "Original / Cleaned GT (control)",
        "Paragraphs": len(cleaned_scope),
        **calculate_scores(cleaned_gt, original_control),
    },
    {
        "Evaluation": "Cleaned / Cleaned GT",
        "Paragraphs": len(cleaned_scope),
        **calculate_scores(cleaned_gt, cleaned_pred),
    },
]).set_index("Evaluation")

print("\n===== ARTICLE-CONTEXT EVALUATION =====")
display(results.round(2))

print("\n===== SHARED-PARAGRAPH PREDICTION COMPARISON =====")

print(
    "Matching:",
    len(original_control & cleaned_pred),
)
print(
    "Original only:",
    len(original_control - cleaned_pred),
)
print(
    "Cleaned only:",
    len(cleaned_pred - original_control),
)

print("\nOriginal prediction files and review CSVs unchanged.")
print("Article-context evaluation completed.")

2026-10-24 09:50:07,740 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, S-ORG, S-MISC, B-PER, E-PER, S-LOC, B-ORG, E-ORG, I-PER, S-PER, B-MISC, I-MISC, E-MISC, I-ORG, B-LOC, E-LOC, I-LOC, <START>, <STOP>
===== ARTICLE-CONTEXT FLERT =====
Model: flair/ner-english-large
Device: cuda:0
Mini batch size: 2

===== ORIGINAL =====
Articles: 30
Paragraphs: 904
Already checkpointed: 0
Remaining: 904


FLERT original: 100%|██████████| 904/904 [00:21<00:00, 41.32paragraph/s]


Inference time: 21.9 seconds
Saved: flert_original_article_context_entities.parquet

===== CLEANED =====
Articles: 30
Paragraphs: 880
Already checkpointed: 0
Remaining: 880


FLERT cleaned: 100%|██████████| 880/880 [00:21<00:00, 40.51paragraph/s]


Inference time: 21.7 seconds
Saved: flert_cleaned_article_context_entities.parquet

===== ARTICLE-CONTEXT EVALUATION =====


,Paragraphs,GT entities,Predicted entities,TP,FP,FN,Precision (%),Recall (%),Micro F1 (%)
Evaluation,,,,,,,,,
Original / Original GT,904,6799,7589,5981,1608,818,78.81,87.97,83.14
Original / Cleaned GT (control),880,6727,7502,5925,1577,802,78.98,88.08,83.28
Cleaned / Cleaned GT,880,6727,7504,5924,1580,803,78.94,88.06,83.25



===== SHARED-PARAGRAPH PREDICTION COMPARISON =====
Matching: 7501
Original only: 1
Cleaned only: 3

Original prediction files and review CSVs unchanged.
Article-context evaluation completed.


Fine Tune

In [32]:
# ============================================================
# 1. PATHS AND CONFIGURATION
# ============================================================

LABELS = {"PER", "ORG", "LOC", "MISC"}

base_dir = (
    Path(REPO_PATH)
    / "ApplyNER"
    / "NER_output"
    / "model_evaluation"
)

split_path = (
    base_dir / "fine_tuning"
    / "flert_article_split.parquet"
)

gt_path = (
    base_dir / "ground_truth"
    / "cohere_entities_cleaned.parquet"
)

issues_path = (
    base_dir / "fine_tuning"
    / "flert_training_alignment_issues.csv"
)

for path in (split_path, gt_path):
    if not path.is_file():
        raise FileNotFoundError(path)

# ============================================================
# 2. LOAD DATA
# ============================================================

split_df = pd.read_parquet(split_path)
gt_df = pd.read_parquet(gt_path)

required_split = {"Source", "Index", "Text", "Split"}
required_gt = {
    "Source", "Index", "Text",
    "word", "start", "end", "entity_group"
}

if not required_split.issubset(split_df.columns):
    raise ValueError("Invalid saved article split.")

if not required_gt.issubset(gt_df.columns):
    raise ValueError("Invalid cleaned Cohere GT.")

if split_df.duplicated(["Source", "Index"]).any():
    raise ValueError("Duplicate paragraph identifiers.")

if set(split_df["Split"]) != {"train", "validation"}:
    raise ValueError("Expected training and validation splits.")

if (split_df.groupby("Source")["Split"].nunique() > 1).any():
    raise ValueError("Article leakage detected.")

def paragraph_key(row):
    return (
        str(row.Source),
        int(row.Index),
        str(row.Text)
    )

scope = {
    paragraph_key(row)
    for row in split_df.itertuples(index=False)
}

# ============================================================
# 3. INDEX COHERE ANNOTATIONS
# ============================================================

annotations = defaultdict(list)

for row in gt_df.itertuples(index=False):

    key = paragraph_key(row)

    if key not in scope:
        continue

    start = int(row.start)
    end = int(row.end)
    label = str(row.entity_group)

    if label not in LABELS:
        raise ValueError(f"Unexpected GT label: {label}")

    if not (0 <= start < end <= len(key[2])):
        raise ValueError(
            f"Invalid GT offsets: {key[0]}, {key[1]}"
        )

    if key[2][start:end] != str(row.word):
        raise ValueError(
            f"GT text mismatch: {key[0]}, {key[1]}"
        )

    annotations[key].append((start, end, label))

# ============================================================
# 4. BUILD TOKEN-ALIGNED FLAIR SENTENCES
# ============================================================

train_sentences = []
dev_sentences = []
issues = []

previous_source = {
    "train": None,
    "validation": None
}

ordered = split_df.sort_values(
    ["Split", "Source", "Index"]
)

for row in ordered.itertuples(index=False):

    key = paragraph_key(row)
    source, index, text = key
    split = str(row.Split)

    gold = sorted(annotations[key])

    sentence = Sentence(text, use_tokenizer=True)

    if not len(sentence):
        issues.append({
            "Source": source,
            "Index": index,
            "Split": split,
            "Reason": "No tokens"
        })
        continue

    start_map = {
        token.start_position: i
        for i, token in enumerate(sentence)
    }

    end_map = {
        token.end_position: i
        for i, token in enumerate(sentence)
    }

    problem = None
    token_spans = []
    previous_end = -1

    # Validate token offsets before adding labels.
    for token in sentence:

        if text[
            token.start_position:token.end_position
        ] != token.text:

            problem = "Tokenizer changed original text"
            break

    if problem is None:

        for start, end, label in gold:

            if start < previous_end:
                problem = "Overlapping GT entities"
                break

            if start not in start_map or end not in end_map:
                problem = "Entity not aligned to token boundaries"
                break

            first = start_map[start]
            last = end_map[end]

            if first > last:
                problem = "Invalid token span"
                break

            token_spans.append((first, last, label))
            previous_end = end

    if problem is not None:

        issues.append({
            "Source": source,
            "Index": index,
            "Split": split,
            "Reason": problem
        })
        continue

    for first, last, label in token_spans:
        sentence[first:last + 1].add_label(
            "ner", label
        )

    # Context must never cross article boundaries.
    sentence.is_document_boundary = (
        previous_source[split] != source
    )

    previous_source[split] = source

    if split == "train":
        train_sentences.append(sentence)
    else:
        dev_sentences.append(sentence)

# ============================================================
# 5. CONSTRUCT THE CORPUS
# ============================================================

if not train_sentences or not dev_sentences:
    raise ValueError("Training or validation corpus is empty.")

flert_training_corpus = Corpus(
    train=train_sentences,
    dev=dev_sentences,
    test=None,
    name="Corpus2GeoRAG_FLERT",
    sample_missing_splits=False
)

# ============================================================
# 6. SAVE ALIGNMENT AUDIT
# ============================================================

issues_df = pd.DataFrame(
    issues,
    columns=["Source", "Index", "Split", "Reason"]
)

if issues_path.exists():
    existing = pd.read_csv(issues_path)

    if not existing.equals(issues_df):
        raise FileExistsError(
            "Existing alignment audit differs. "
            "It was not overwritten."
        )
else:
    issues_df.to_csv(
        issues_path,
        index=False,
        encoding="utf-8-sig"
    )

# ============================================================
# 7. SUMMARY
# ============================================================

print("===== FLERT TRAINING DATA =====")
print(f"Original training paragraphs: {(split_df['Split'] == 'train').sum():,}")
print(f"Usable training paragraphs: {len(train_sentences):,}")
print(f"Original validation paragraphs: {(split_df['Split'] == 'validation').sum():,}")
print(f"Usable validation paragraphs: {len(dev_sentences):,}")
print(f"Excluded for token alignment: {len(issues):,}")

print("\nCorpus:", flert_training_corpus)

if issues:
    print("\nAlignment issues:")
    print(issues_df["Reason"].value_counts())
    print("Details:", issues_path)

print("\nTraining corpus is ready.")
print("Original Cohere annotations remain unchanged.")

===== FLERT TRAINING DATA =====
Original training paragraphs: 707
Usable training paragraphs: 672
Original validation paragraphs: 173
Usable validation paragraphs: 169
Excluded for token alignment: 39

Corpus: Corpus: 672 train + 169 dev + 0 test sentences

Alignment issues:
Reason
Entity not aligned to token boundaries    39
Name: count, dtype: int64
Details: C:\Users\idosc\Corpus2GeoRAG\ApplyNER\NER_output\model_evaluation\fine_tuning\flert_training_alignment_issues.csv

Training corpus is ready.
Original Cohere annotations remain unchanged.


In [33]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_NAME = "flair/ner-english-large"
SEED = 42

EPOCHS = 2
LEARNING_RATE = 5e-6
BATCH_SIZE = 2
BATCH_CHUNK_SIZE = 1

output_dir = (
    Path(REPO_PATH)
    / "ApplyNER"
    / "NER_output"
    / "model_evaluation"
    / "fine_tuning"
    / "flert_stage1"
)

if "flert_training_corpus" not in globals():
    raise RuntimeError(
        "Run the FLERT training corpus cell first."
    )

# Never silently restart training over a saved model.
if output_dir.exists() and any(output_dir.iterdir()):
    raise FileExistsError(
        f"Training directory already contains files: {output_dir}\n"
        "Inspect the existing checkpoints before rerunning."
    )

# ============================================================
# 2. REPRODUCIBILITY AND DEVICE
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

flair.device = device

print("===== FLERT FINE-TUNING =====")

print("Model:", MODEL_NAME)
print("Device:", device)
print("Training paragraphs:", len(flert_training_corpus.train))
print("Validation paragraphs:", len(flert_training_corpus.dev))
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)

# ============================================================
# 3. LOAD ORIGINAL PRETRAINED FLERT
# ============================================================

model = SequenceTagger.load(MODEL_NAME)
model.to(device)

# ============================================================
# 4. INITIALIZE TRAINER
# ============================================================

trainer = ModelTrainer(
    model,
    flert_training_corpus
)

output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================
# 5. FINE-TUNE
# ============================================================

training_results = trainer.fine_tune(
    base_path=output_dir,
    learning_rate=LEARNING_RATE,
    mini_batch_size=BATCH_SIZE,
    mini_batch_chunk_size=BATCH_CHUNK_SIZE,
    eval_batch_size=2,
    max_epochs=EPOCHS,
    use_amp=(device.type == "cuda"),
    embeddings_storage_mode="none",
    save_model_each_k_epochs=1,
    save_final_model=True,
    use_final_model_for_eval=False,
)

# ============================================================
# 6. RESULTS
# ============================================================

print("\n===== FINE-TUNING COMPLETED =====")
print("Output directory:", output_dir)

print("\nSaved files:")
for path in sorted(output_dir.iterdir()):
    print(" ", path.name)

print("\nStage 1 fine-tuning completed.")
print("The original FLERT model was not overwritten.")

===== FLERT FINE-TUNING =====
Model: flair/ner-english-large
Device: cuda:0
Training paragraphs: 672
Validation paragraphs: 169
Epochs: 2
Learning rate: 5e-06
2026-10-24 10:05:56,871 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, S-ORG, S-MISC, B-PER, E-PER, S-LOC, B-ORG, E-ORG, I-PER, S-PER, B-MISC, I-MISC, E-MISC, I-ORG, B-LOC, E-LOC, I-LOC, <START>, <STOP>
2026-10-24 10:05:57,438 ----------------------------------------------------------------------------------------------------
2026-10-24 10:05:57,441 Model: "SequenceTagger(
  (embeddings): TransformerWordEmbeddings(
    (model): XLMRobertaModel(
      (embeddings): XLMRobertaEmbeddings(
        (word_embeddings): Embedding(250002, 1024, padding_idx=1)
        (position_embeddings): Embedding(514, 1024, padding_idx=1)
        (token_type_embeddings): Embedding(1, 1024)
        (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (e

C:\Users\idosc\.venvs\corpus2georag\Lib\site-packages\flair\trainers\trainer.py:545: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp and flair.device.type != "cpu")


2026-10-24 10:07:23,687 epoch 1 - iter 33/336 - loss 0.50766405 - time (sec): 86.22 - samples/sec: 54.57 - lr: 0.000002 - momentum: 0.000000
2026-10-24 10:08:56,822 epoch 1 - iter 66/336 - loss 0.43635628 - time (sec): 179.36 - samples/sec: 58.17 - lr: 0.000004 - momentum: 0.000000
2026-10-24 10:10:17,517 epoch 1 - iter 99/336 - loss 0.42026386 - time (sec): 260.05 - samples/sec: 55.32 - lr: 0.000005 - momentum: 0.000000
2026-10-24 10:11:42,320 epoch 1 - iter 132/336 - loss 0.38119866 - time (sec): 344.85 - samples/sec: 56.27 - lr: 0.000005 - momentum: 0.000000
2026-10-24 10:13:19,210 epoch 1 - iter 165/336 - loss 0.34752679 - time (sec): 441.74 - samples/sec: 53.89 - lr: 0.000004 - momentum: 0.000000
2026-10-24 10:15:02,844 epoch 1 - iter 198/336 - loss 0.31588584 - time (sec): 545.38 - samples/sec: 54.39 - lr: 0.000004 - momentum: 0.000000
2026-10-24 10:16:37,760 epoch 1 - iter 231/336 - loss 0.28746249 - time (sec): 640.29 - samples/sec: 54.73 - lr: 0.000004 - momentum: 0.000000
202

100%|██████████| 85/85 [00:28<00:00,  2.97it/s]


2026-10-24 10:21:52,291 DEV : loss 0.266544908285141 - f1-score (micro avg)  0.7985
2026-10-24 10:21:52,295 saving best model
2026-10-24 10:21:55,363 ----------------------------------------------------------------------------------------------------
2026-10-24 10:23:15,387 epoch 2 - iter 33/336 - loss 0.12203193 - time (sec): 80.02 - samples/sec: 56.45 - lr: 0.000003 - momentum: 0.000000
2026-10-24 10:24:40,148 epoch 2 - iter 66/336 - loss 0.13849087 - time (sec): 164.78 - samples/sec: 55.43 - lr: 0.000002 - momentum: 0.000000
2026-10-24 10:26:16,119 epoch 2 - iter 99/336 - loss 0.12402039 - time (sec): 260.75 - samples/sec: 53.80 - lr: 0.000002 - momentum: 0.000000
2026-10-24 10:27:35,573 epoch 2 - iter 132/336 - loss 0.11806810 - time (sec): 340.20 - samples/sec: 56.06 - lr: 0.000002 - momentum: 0.000000
2026-10-24 10:28:54,173 epoch 2 - iter 165/336 - loss 0.12876133 - time (sec): 418.80 - samples/sec: 56.31 - lr: 0.000002 - momentum: 0.000000
2026-10-24 10:30:19,921 epoch 2 - iter

100%|██████████| 85/85 [00:33<00:00,  2.52it/s]

2026-10-24 10:36:55,062 DEV : loss 0.27476751804351807 - f1-score (micro avg)  0.8011
2026-10-24 10:36:55,065 saving best model


2026-10-24 10:37:05,619 Loading model from best epoch ...
2026-10-24 10:37:18,663 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, S-ORG, S-MISC, B-PER, E-PER, S-LOC, B-ORG, E-ORG, I-PER, S-PER, B-MISC, I-MISC, E-MISC, I-ORG, B-LOC, E-LOC, I-LOC, <START>, <STOP>
2026-10-24 10:37:19,318 Test data not provided setting final score to 0

===== FINE-TUNING COMPLETED =====
Output directory: C:\Users\idosc\Corpus2GeoRAG\ApplyNER\NER_output\model_evaluation\fine_tuning\flert_stage1

Saved files:
  best-model.pt
  dev.tsv
  final-model.pt
  loss.tsv
  model_epoch_1.pt
  model_epoch_2.pt
  training.log

Stage 1 fine-tuning completed.
The original FLERT model was not overwritten.


Evaluate fine tuned model

In [35]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

LABELS = ("PER", "ORG", "LOC", "MISC")
BATCH_SIZE = 2

evaluation_dir = (
    Path(REPO_PATH)
    / "ApplyNER"
    / "NER_output"
    / "model_evaluation"
)

model_dir = evaluation_dir / "NER_models_evaluation"
gt_dir = evaluation_dir / "ground_truth"
fine_tuning_dir = evaluation_dir / "fine_tuning"

stage1_dir = fine_tuning_dir / "flert_stage1"

split_path = fine_tuning_dir / "flert_article_split.parquet"
gt_path = gt_dir / "cohere_entities_cleaned.parquet"

baseline_path = (
    model_dir
    / "flert_cleaned_article_context_entities.parquet"
)

baseline_checkpoint = (
    model_dir
    / "flert_cleaned_article_context_raw.jsonl"
)

best_model_path = stage1_dir / "best-model.pt"
final_model_path = stage1_dir / "final-model.pt"

prediction_output = (
    stage1_dir / "flert_stage1_validation_predictions.parquet"
)

comparison_output = (
    stage1_dir / "flert_stage1_validation_comparison.csv"
)

per_label_output = (
    stage1_dir / "flert_stage1_validation_per_label.csv"
)

for path in [
    split_path, gt_path, baseline_path, baseline_checkpoint
]:
    if not path.is_file():
        raise FileNotFoundError(path)

if best_model_path.is_file():
    model_path = best_model_path
elif final_model_path.is_file():
    model_path = final_model_path
else:
    raise FileNotFoundError(
        "No fine-tuned FLERT checkpoint found in "
        f"{stage1_dir}"
    )

# ============================================================
# 2. LOAD THE FIXED VALIDATION DATASET
# ============================================================

split_df = pd.read_parquet(split_path)

required_split = {"Source", "Index", "Text", "Split"}

if not required_split.issubset(split_df.columns):
    raise ValueError("Invalid train/validation split.")

if split_df.duplicated(["Source", "Index"]).any():
    raise ValueError("Duplicate paragraph identifiers.")

if (split_df.groupby("Source")["Split"].nunique() > 1).any():
    raise ValueError("Article leakage between splits.")

validation_df = (
    split_df.loc[split_df["Split"] == "validation"]
    .sort_values(["Source", "Index"])
    .reset_index(drop=True)
)

if validation_df.empty:
    raise ValueError("Validation dataset is empty.")

def paragraph_key(row):
    return (
        str(row.Source),
        int(row.Index),
        str(row.Text)
    )

validation_scope = {
    paragraph_key(row)
    for row in validation_df.itertuples(index=False)
}

validation_sources = set(validation_df["Source"])

# ============================================================
# 3. LOAD GROUND TRUTH AND ORIGINAL FLERT PREDICTIONS
# ============================================================

gt_df = pd.read_parquet(gt_path)
baseline_df = pd.read_parquet(baseline_path)

required_entity_columns = {
    "Source", "Index", "Text", "word",
    "start", "end", "entity_group"
}

for name, df in [
    ("Cohere GT", gt_df),
    ("Original FLERT", baseline_df)
]:
    missing = required_entity_columns - set(df.columns)

    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

# Confirm the original model actually processed all
# validation paragraphs, including zero-entity paragraphs.

baseline_completed = set()

with baseline_checkpoint.open("r", encoding="utf-8") as f:
    for line in f:

        if not line.strip():
            continue

        record = json.loads(line)

        key = (
            str(record["Source"]),
            int(record["Index"]),
            str(record["Text"])
        )

        if key not in validation_scope:
            continue

        settings = record.get("settings", {})

        if (
            settings.get("model") != "flair/ner-english-large"
            or settings.get("context") != "article"
        ):
            raise ValueError(
                "Original baseline uses unexpected settings."
            )

        baseline_completed.add(key)

if baseline_completed != validation_scope:
    raise ValueError(
        "Original FLERT baseline does not cover "
        "all validation paragraphs."
    )

# ============================================================
# 4. BUILD EXACT-MATCH ENTITY SETS
# ============================================================

def entity_set(df, name):

    result = set()

    for row in df.itertuples(index=False):

        key = paragraph_key(row)

        if key not in validation_scope:
            continue

        start = int(row.start)
        end = int(row.end)
        label = str(row.entity_group)
        word = str(row.word)

        if label not in LABELS:
            raise ValueError(f"{name}: unexpected label {label}")

        if not (0 <= start < end <= len(key[2])):
            raise ValueError(f"{name}: invalid entity offsets.")

        if key[2][start:end] != word:
            raise ValueError(f"{name}: word/offset mismatch.")

        item = (*key, start, end, label)

        if item in result:
            raise ValueError(f"{name}: duplicate annotation.")

        result.add(item)

    return result


ground_truth = entity_set(gt_df, "Cohere GT")
original_predictions = entity_set(
    baseline_df, "Original FLERT"
)

# ============================================================
# 5. RELEASE TRAINING MODEL REFERENCES
# ============================================================

# The completed training checkpoints are saved on disk.
# Release notebook references to reduce GPU memory usage.

if "trainer" in globals():
    del trainer

if "model" in globals():
    del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

flair.device = device

# ============================================================
# 6. LOAD THE FINE-TUNED MODEL
# ============================================================

fine_tuned_model = SequenceTagger.load(str(model_path))
fine_tuned_model.to(device)
fine_tuned_model.eval()

print("===== FLERT VALIDATION COMPARISON =====")

print("Original model: flair/ner-english-large")
print("Fine-tuned checkpoint:", model_path.name)
print("Device:", device)
print("Validation articles:", len(validation_sources))
print("Validation paragraphs:", len(validation_scope))
print("GT entities:", len(ground_truth))
print("Original FLERT predictions:", len(original_predictions))

# ============================================================
# 7. RUN FINE-TUNED FLERT ON VALIDATION ARTICLES
# ============================================================

articles = defaultdict(list)

for row in validation_df.itertuples(index=False):
    articles[str(row.Source)].append(paragraph_key(row))

prediction_rows = []

with tqdm(
    total=len(validation_scope),
    desc="Fine-tuned FLERT validation",
    unit="paragraph"
) as progress:

    for source in sorted(articles):

        keys = articles[source]

        sentences = [
            Sentence(text, use_tokenizer=True)
            for _, _, text in keys
        ]

        if any(len(sentence) == 0 for sentence in sentences):
            raise ValueError(
                f"Empty tokenized paragraph in {source}"
            )

        # Preserve article-level context.
        # Never connect unrelated Wikipedia articles.
        sentences[0].is_document_boundary = True

        with torch.no_grad():
            fine_tuned_model.predict(
                sentences,
                mini_batch_size=BATCH_SIZE,
                verbose=False,
                embedding_storage_mode="none"
            )

        for key, sentence in zip(keys, sentences):

            for span in sentence.get_spans("ner"):

                label = span.get_label("ner")
                start = int(span.start_position)
                end = int(span.end_position)

                if not (0 <= start < end <= len(key[2])):
                    raise ValueError(
                        f"Invalid prediction span: "
                        f"{key[0]}, {key[1]}"
                    )

                prediction_rows.append({
                    "Source": key[0],
                    "Index": key[1],
                    "Text": key[2],
                    "entity_group": str(label.value),
                    "score": float(label.score),
                    "word": key[2][start:end],
                    "start": start,
                    "end": end
                })

            progress.update(1)

# ============================================================
# 8. SAVE FINE-TUNED PREDICTIONS
# ============================================================

fine_tuned_df = pd.DataFrame(
    prediction_rows,
    columns=[
        "Source", "Index", "Text",
        "entity_group", "score",
        "word", "start", "end"
    ]
)

fine_tuned_predictions = entity_set(
    fine_tuned_df,
    "Fine-tuned FLERT"
)

fine_tuned_df.to_parquet(
    prediction_output,
    engine="pyarrow",
    compression="zstd",
    index=False
)

# ============================================================
# 9. CALCULATE PERFORMANCE METRICS
# ============================================================

def calculate_metrics(truth, predictions):

    tp_entities = truth & predictions
    fp_entities = predictions - truth
    fn_entities = truth - predictions

    def scores(tp, fp, fn):

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        return precision * 100, recall * 100, f1 * 100

    tp = len(tp_entities)
    fp = len(fp_entities)
    fn = len(fn_entities)

    precision, recall, micro_f1 = scores(tp, fp, fn)

    per_label = []

    for label in LABELS:

        label_tp = sum(
            entity[-1] == label for entity in tp_entities
        )

        label_fp = sum(
            entity[-1] == label for entity in fp_entities
        )

        label_fn = sum(
            entity[-1] == label for entity in fn_entities
        )

        p, r, f1 = scores(
            label_tp, label_fp, label_fn
        )

        per_label.append({
            "Label": label,
            "Support": label_tp + label_fn,
            "TP": label_tp,
            "FP": label_fp,
            "FN": label_fn,
            "Precision (%)": p,
            "Recall (%)": r,
            "F1 (%)": f1
        })

    macro_f1 = sum(
        row["F1 (%)"] for row in per_label
    ) / len(LABELS)

    overall = {
        "GT entities": len(truth),
        "Predicted entities": len(predictions),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision (%)": precision,
        "Recall (%)": recall,
        "Micro F1 (%)": micro_f1,
        "Macro F1 (%)": macro_f1
    }

    return overall, per_label

original_metrics, original_per_label = calculate_metrics(
    ground_truth,
    original_predictions
)

fine_tuned_metrics, fine_tuned_per_label = calculate_metrics(
    ground_truth,
    fine_tuned_predictions
)

# ============================================================
# 10. DISPLAY OVERALL COMPARISON
# ============================================================

comparison = pd.DataFrame([
    {
        "Model": "Original FLERT",
        "Paragraphs": len(validation_scope),
        **original_metrics
    },
    {
        "Model": "Fine-tuned FLERT",
        "Paragraphs": len(validation_scope),
        **fine_tuned_metrics
    }
]).set_index("Model")

print("\n===== OVERALL VALIDATION METRICS =====")
display(comparison.round(2))

comparison.to_csv(
    comparison_output,
    encoding="utf-8-sig"
)

# ============================================================
# 11. PERFORMANCE CHANGE
# ============================================================

metric_columns = [
    "Precision (%)",
    "Recall (%)",
    "Micro F1 (%)",
    "Macro F1 (%)"
]

changes = pd.DataFrame({
    "Metric": metric_columns,
    "Original (%)": [
        original_metrics[m] for m in metric_columns
    ],
    "Fine-tuned (%)": [
        fine_tuned_metrics[m] for m in metric_columns
    ],
    "Difference (pp)": [
        fine_tuned_metrics[m] - original_metrics[m]
        for m in metric_columns
    ]
})

print("\n===== PERFORMANCE CHANGE =====")
display(changes.round(2))

# ============================================================
# 12. PER-ENTITY-TYPE COMPARISON
# ============================================================

per_label_comparison = pd.concat([
    pd.DataFrame(original_per_label).assign(
        Model="Original FLERT"
    ),
    pd.DataFrame(fine_tuned_per_label).assign(
        Model="Fine-tuned FLERT"
    )
], ignore_index=True)

print("\n===== PER-ENTITY-TYPE COMPARISON =====")

display(
    per_label_comparison[
        [
            "Model",
            "Label",
            "Support",
            "Precision (%)",
            "Recall (%)",
            "F1 (%)"
        ]
    ].round(2)
)

per_label_comparison.to_csv(
    per_label_output,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 13. HOW MANY ERRORS WERE CORRECTED?
# ============================================================

original_correct = ground_truth & original_predictions
fine_tuned_correct = ground_truth & fine_tuned_predictions

original_errors = original_predictions - ground_truth
fine_tuned_errors = fine_tuned_predictions - ground_truth

newly_correct = fine_tuned_correct - original_correct
lost_correct = original_correct - fine_tuned_correct

removed_false_positives = (
    original_errors - fine_tuned_errors
)

introduced_false_positives = (
    fine_tuned_errors - original_errors
)

print("\n===== ENTITY-LEVEL CHANGES =====")

print(
    "Previously missed/wrong GT entities now correct:",
    len(newly_correct)
)

print(
    "Previously correct GT entities now incorrect:",
    len(lost_correct)
)

print(
    "Original false positives eliminated:",
    len(removed_false_positives)
)

print(
    "New false positives introduced:",
    len(introduced_false_positives)
)

print("\n===== SAVED RESULTS =====")
print("Fine-tuned predictions:", prediction_output)
print("Overall comparison:", comparison_output)
print("Per-label comparison:", per_label_output)

print("\nValidation completed.")
print("Both models evaluated on the same held-out articles.")
print("Original FLERT and Cohere datasets remain unchanged.")

2026-10-24 10:52:37,903 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, S-ORG, S-MISC, B-PER, E-PER, S-LOC, B-ORG, E-ORG, I-PER, S-PER, B-MISC, I-MISC, E-MISC, I-ORG, B-LOC, E-LOC, I-LOC, <START>, <STOP>
===== FLERT VALIDATION COMPARISON =====
Original model: flair/ner-english-large
Fine-tuned checkpoint: best-model.pt
Device: cuda:0
Validation articles: 6
Validation paragraphs: 173
GT entities: 944
Original FLERT predictions: 1033


Fine-tuned FLERT validation: 100%|██████████| 173/173 [00:03<00:00, 48.80paragraph/s]


===== OVERALL VALIDATION METRICS =====


,Paragraphs,GT entities,Predicted entities,TP,FP,FN,Precision (%),Recall (%),Micro F1 (%),Macro F1 (%)
Model,,,,,,,,,,
Original FLERT,173,944,1033,784,249,160,75.90,83.05,79.31,80.86
Fine-tuned FLERT,173,944,974,764,210,180,78.44,80.93,79.67,81.08



===== PERFORMANCE CHANGE =====


,Metric,Original (%),Fine-tuned (%),Difference (pp)
0,Precision (%),75.90,78.44,2.54
1,Recall (%),83.05,80.93,-2.12
2,Micro F1 (%),79.31,79.67,0.35
3,Macro F1 (%),80.86,81.08,0.22



===== PER-ENTITY-TYPE COMPARISON =====


,Model,Label,Support,Precision (%),Recall (%),F1 (%)
0,Original FLERT,PER,52,89.09,94.23,91.59
1,Original FLERT,ORG,244,74.04,71.31,72.65
2,Original FLERT,LOC,385,84.18,89.87,86.93
3,Original FLERT,MISC,263,64.76,81.75,72.27
4,Fine-tuned FLERT,PER,52,88.89,92.31,90.57
5,Fine-tuned FLERT,ORG,244,71.80,78.28,74.90
6,Fine-tuned FLERT,LOC,385,83.17,87.27,85.17
7,Fine-tuned FLERT,MISC,263,75.60,71.86,73.68



===== ENTITY-LEVEL CHANGES =====
Previously missed/wrong GT entities now correct: 30
Previously correct GT entities now incorrect: 50
Original false positives eliminated: 104
New false positives introduced: 65

===== SAVED RESULTS =====
Fine-tuned predictions: C:\Users\idosc\Corpus2GeoRAG\ApplyNER\NER_output\model_evaluation\fine_tuning\flert_stage1\flert_stage1_validation_predictions.parquet
Overall comparison: C:\Users\idosc\Corpus2GeoRAG\ApplyNER\NER_output\model_evaluation\fine_tuning\flert_stage1\flert_stage1_validation_comparison.csv
Per-label comparison: C:\Users\idosc\Corpus2GeoRAG\ApplyNER\NER_output\model_evaluation\fine_tuning\flert_stage1\flert_stage1_validation_per_label.csv

Validation completed.
Both models evaluated on the same held-out articles.
Original FLERT and Cohere datasets remain unchanged.


### Commit to Repo

In [23]:
GITHUB_USER = "idoschw3"
GITHUB_REPO = "Corpus2GeoRAG"
GITHUB_EMAIL = "193781032+idoschw3@users.noreply.github.com"
COMMIT_MESSAGE = "Update NER workflow"

LOCAL_REPO_PATH = Path.home() / GITHUB_REPO
MAX_GIT_FILE_BYTES = 100 * 1024 * 1024

# --------------------------------------------------
# Detect runtime
# --------------------------------------------------

if os.name == "nt":
    IS_LOCAL = True
elif os.environ.get("COLAB_RELEASE_TAG"):
    IS_LOCAL = False
else:
    raise RuntimeError(
        "Could not identify this runtime as Windows or a hosted Colab VM."
    )

if shutil.which("git") is None:
    raise RuntimeError(
        "Git is not available to this Jupyter environment. "
        "Install Git for Windows and restart Jupyter if running locally."
    )

# --------------------------------------------------
# Locate the repository
# --------------------------------------------------

if IS_LOCAL:
    candidates = [Path.cwd(), LOCAL_REPO_PATH]

    previous_repo_path = globals().get("REPO_PATH")
    if previous_repo_path:
        candidates.insert(0, Path(previous_repo_path))

    detected_repo = None

    for candidate in candidates:
        if not candidate.is_dir():
            continue

        probe = subprocess.run(
            ["git", "rev-parse", "--show-toplevel"],
            cwd=candidate,
            text=True,
            encoding="utf-8",
            errors="replace",
            capture_output=True
        )

        if probe.returncode == 0:
            root = Path(probe.stdout.strip()).resolve()

            if root.name.lower() == GITHUB_REPO.lower():
                detected_repo = root
                break

    if detected_repo is None:
        raise FileNotFoundError(
            f"Could not find your local {GITHUB_REPO} checkout.\n"
            f"Clone it to {LOCAL_REPO_PATH}, or update LOCAL_REPO_PATH."
        )

    REPO_PATH = str(detected_repo)

else:
    REPO_PATH = f"/content/{GITHUB_REPO}"

    if not Path(REPO_PATH).is_dir():
        raise FileNotFoundError(
            f"{REPO_PATH} does not exist. Run your clone cell first."
        )

os.chdir(REPO_PATH)

print("Runtime:", "Local Windows" if IS_LOCAL else "Colab VM")
print("Repository:", REPO_PATH)


def run_git(args, *, env=None, show_output=True):
    """Run Git and include its actual output if it fails."""
    result = subprocess.run(
        ["git", *args],
        cwd=REPO_PATH,
        env=env,
        stdin=subprocess.DEVNULL,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True
    )

    if show_output:
        if result.stdout:
            print(result.stdout, end="")
        if result.stderr:
            print(result.stderr, end="")

    if result.returncode != 0:
        details = (result.stderr or result.stdout or "").strip()
        raise RuntimeError(f"git {' '.join(args)} failed:\n{details}")

    return result


# --------------------------------------------------
# Git configuration
# --------------------------------------------------

run_git(["config", "user.name", GITHUB_USER])
run_git(["config", "user.email", GITHUB_EMAIL])
run_git([
    "remote", "set-url", "origin",
    f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
])

branch = run_git(
    ["branch", "--show-current"],
    show_output=False
).stdout.strip()

if not branch:
    raise RuntimeError("Could not determine the current Git branch.")

print("Branch:", branch)

# Stop if an earlier rebase or merge is still paused.
for state_path in ("rebase-merge", "rebase-apply", "MERGE_HEAD"):
    git_state = run_git(
        ["rev-parse", "--git-path", state_path],
        show_output=False
    ).stdout.strip()

    if Path(git_state).exists():
        raise RuntimeError(
            "An earlier rebase or merge is still in progress. "
            "Check `git status` and resolve it before rerunning this cell."
        )

# --------------------------------------------------
# Authentication and synchronization
# --------------------------------------------------

git_env = os.environ.copy()
git_env["GIT_TERMINAL_PROMPT"] = "0"

auth_options = []
askpass_dir = None

try:
    if IS_LOCAL:
        # Use Git credentials already configured on your computer.
        print("Authentication: local Git credentials.")

    else:
        from google.colab import userdata

        github_token = userdata.get("GITHUB_TOKEN")

        if not github_token:
            raise ValueError(
                "GITHUB_TOKEN is missing from Colab Secrets. "
                "Add it and enable notebook access."
            )

        askpass_dir = tempfile.TemporaryDirectory(
            prefix="github-askpass-"
        )
        askpass_path = Path(askpass_dir.name) / "askpass.sh"

        askpass_path.write_text(
            "#!/bin/sh\n"
            'case "$1" in\n'
            '  *Username*) printf "%s\\n" "$GITHUB_USER" ;;\n'
            '  *Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
            "esac\n",
            encoding="utf-8"
        )
        askpass_path.chmod(0o700)

        git_env["GITHUB_USER"] = GITHUB_USER
        git_env["GITHUB_TOKEN"] = github_token
        git_env["GIT_ASKPASS"] = str(askpass_path)

        auth_options = ["-c", "credential.helper="]

        print("Authentication: Colab GITHUB_TOKEN Secret.")

    # --------------------------------------------------
    # Prevent accidental nested repository
    # --------------------------------------------------

    ignore_entry = f"/{GITHUB_REPO}/"
    gitignore_path = Path(REPO_PATH) / ".gitignore"

    existing_ignore = (
        gitignore_path.read_text(encoding="utf-8")
        if gitignore_path.exists()
        else ""
    )

    if ignore_entry not in existing_ignore.splitlines():
        with gitignore_path.open("a", encoding="utf-8") as f:
            if existing_ignore and not existing_ignore.endswith("\n"):
                f.write("\n")
            f.write(ignore_entry + "\n")

    nested_repo = run_git(
        ["ls-files", "--stage", "--", GITHUB_REPO],
        show_output=False
    ).stdout.strip()

    if nested_repo:
        run_git([
            "rm", "--cached", "-r", "-f", "--", GITHUB_REPO
        ])
        print("Removed the nested repository path from Git tracking.")

    # --------------------------------------------------
    # Find files exceeding GitHub's per-file limit
    # --------------------------------------------------

    candidate_files = run_git(
        [
            "ls-files",
            "--cached",
            "--others",
            "--exclude-standard",
            "-z"
        ],
        show_output=False
    ).stdout.split("\0")

    excluded_files = set()

    for relative_path in sorted(set(candidate_files) - {""}):
        file_path = Path(REPO_PATH) / relative_path

        if not file_path.is_file():
            continue

        file_size = file_path.stat().st_size

        if file_size > MAX_GIT_FILE_BYTES:
            excluded_files.add(relative_path)

            # Remove from Git's index, while keeping the file on disk.
            run_git(
                [
                    "rm",
                    "--cached",
                    "--ignore-unmatch",
                    "-f",
                    "--",
                    f":(top,literal){relative_path}"
                ],
                show_output=False
            )

            print(
                f"Excluded oversized file: {relative_path} "
                f"({file_size / 1024**2:.2f} MiB). "
                "Kept on disk."
            )

    # --------------------------------------------------
    # Stage changes, excluding oversized files
    # --------------------------------------------------

    exclusion_pathspecs = [
        f":(top,literal,exclude){path}"
        for path in sorted(excluded_files)
    ]

    run_git(["add", "-A", "--", ".", *exclusion_pathspecs])

    # Exclude notebook copies, preserving the original cell's behavior.
    staged_files = run_git(
        ["diff", "--cached", "--name-only", "-z"],
        show_output=False
    ).stdout.split("\0")

    for notebook in staged_files:
        if (
            notebook.endswith(".ipynb")
            and notebook not in excluded_files
        ):
            run_git([
                "restore", "--staged", "--",
                f":(top,literal){notebook}"
            ])

    # --------------------------------------------------
    # Create a normal commit
    # --------------------------------------------------

    staged_diff = subprocess.run(
        ["git", "diff", "--cached", "--quiet"],
        cwd=REPO_PATH,
        stdin=subprocess.DEVNULL
    )

    if staged_diff.returncode == 1:
        run_git(["commit", "-m", COMMIT_MESSAGE])
        print("Repository changes committed.")
    elif staged_diff.returncode == 0:
        print("No new repository changes to commit.")
    else:
        raise RuntimeError("Could not check the staged changes.")

    # --------------------------------------------------
    # Restore tracked notebook copies on disk
    # This discards edits to those on-disk copies.
    # --------------------------------------------------

    tracked_files = run_git(
        ["ls-tree", "-r", "--name-only", "-z", "HEAD"],
        show_output=False
    ).stdout.split("\0")

    for notebook in tracked_files:
        if notebook.endswith(".ipynb"):
            run_git([
                "restore", "--worktree", "--",
                f":(top,literal){notebook}"
            ])

    # --------------------------------------------------
    # Pull/rebase GitHub changes
    # --------------------------------------------------

    pull = subprocess.run(
        ["git", *auth_options, "pull", "--rebase", "origin", branch],
        cwd=REPO_PATH,
        env=git_env,
        stdin=subprocess.DEVNULL,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True
    )

    if pull.stdout:
        print(pull.stdout, end="")
    if pull.stderr:
        print(pull.stderr, end="")

    if pull.returncode != 0:
        conflicted = run_git(
            ["diff", "--name-only", "--diff-filter=U"],
            show_output=False
        ).stdout.splitlines()

        if conflicted == [".gitignore"]:
            ours = run_git(
                ["show", ":2:.gitignore"],
                show_output=False
            ).stdout

            theirs = run_git(
                ["show", ":3:.gitignore"],
                show_output=False
            ).stdout

            if ours.strip() != theirs.strip():
                raise RuntimeError(
                    "The .gitignore versions differ. The rebase is paused; "
                    "inspect the conflict before continuing."
                )

            gitignore_path.write_text(
                ours + ("\n" if ours and not ours.endswith("\n") else ""),
                encoding="utf-8"
            )

            run_git(["add", "--", ".gitignore"])
            run_git([
                "-c", "core.editor=true", "rebase", "--continue"
            ])

        else:
            raise RuntimeError(
                "Pull failed. Check the Git output above and `git status`. "
                "If running locally, verify your GitHub authentication."
            )

    # --------------------------------------------------
    # Push synchronized changes
    # --------------------------------------------------

    run_git(
        [*auth_options, "push", "origin", branch],
        env=git_env
    )

    print(
        f"Successfully pushed to {GITHUB_USER}/{GITHUB_REPO} "
        f"on branch '{branch}'."
    )

finally:
    if askpass_dir is not None:
        askpass_dir.cleanup()

    git_env.pop("GITHUB_TOKEN", None)

Runtime: Local Windows
Repository: C:\Users\idosc\Corpus2GeoRAG
Branch: main
Authentication: local Git credentials.
[main edc0bd6] Update NER workflow
 2 files changed, 880 insertions(+)
 create mode 100644 ApplyNER/NER_output/model_evaluation/NER_models_evaluation/flair_cleaned_raw.jsonl
 create mode 100644 ApplyNER/NER_output/model_evaluation/NER_models_evaluation/flair_entities_cleaned.parquet
Repository changes committed.
Current branch main is up to date.
From https://github.com/idoschw3/Corpus2GeoRAG
 * branch            main       -> FETCH_HEAD
To https://github.com/idoschw3/Corpus2GeoRAG.git
   f56c257..edc0bd6  main -> main
Successfully pushed to idoschw3/Corpus2GeoRAG on branch 'main'.
